<a href="https://colab.research.google.com/github/NealItharaja/Icebreaker/blob/main/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## LibreLane Colab

This Google Colab notebook will:
* Install LibreLane and its dependencies
* Run a simple design, namely a serial-parallel multiplier, through the flow
  and targeting the [open source sky130 PDK](https://github.com/google/skywater-pdk/)
  by Google and Skywater.

In [3]:
# @title Setup Nix {display-mode: "form"}
# @markdown <img src="https://raw.githubusercontent.com/NixOS/nixos-artwork/51a27e4a011e95cb559e37d32c44cf89b50f5154/logo/nix-snowflake-colours.svg" width="32"/>
# @markdown
# @markdown Nix is a package manager with an emphasis on reproducible builds,
# @markdown and it is the primary method for installing LibreLane.
# @markdown
# @markdown This step installs the Nix package manager and enables the
# @markdown FOSSi Foundation Nix Cache.
# @markdown
# @markdown If you're not in a Colab, this just sets the environment variables.
# @markdown You will need to install Nix and enable flakes on your own following
# @markdown [this guide](https://librelane.readthedocs.io/en/stable/getting_started/common/nix_installation/index.html).
import os
from pathlib import Path
import subprocess
import sys
import shutil
import tempfile

os.environ["LOCALE_ARCHIVE"] = "/usr/lib/locale/locale-archive"

if "google.colab" in sys.modules:
    if shutil.which("nix-env") is None:
        with tempfile.TemporaryDirectory() as d:
            d = Path(d)
            installer_path = d / "nix"
            !curl --proto '=https' --tlsv1.2 -sSf -L https://install.determinate.systems/nix > {installer_path}
            with subprocess.Popen(
                [
                    "bash",
                    installer_path,
                    "install",
                    "--prefer-upstream-nix",
                    "--no-confirm",
                    "--extra-conf",
                    "extra-substituters = https://nix-cache.fossi-foundation.org\nextra-trusted-public-keys = nix-cache.fossi-foundation.org:3+K59iFwXqKsL7BNu6Guy0v+uTlwsxYQxjspXzqLYQs=\n",
                ],
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                encoding="utf8",
            ) as p:
                for line in p.stdout:
                    print(line, end="")
else:
    if shutil.which("nix-env") is None:
        raise RuntimeError("Nix is not installed!")

os.environ["PATH"] = f"/nix/var/nix/profiles/default/bin/:{os.getenv('PATH')}"

info: downloading the Determinate Nix Installer
 INFO nix-installer v3.21.9
 INFO Step: Create directory `/nix`
 INFO Step: Provision Nix
 INFO Step: Create build users (UID 30001-30032) and group (GID 30000)
 INFO Step: Configure Nix
 INFO Step: Create directory `/etc/tmpfiles.d`
 INFO Step: Configure upstream Nix daemon service
 INFO Step: Cleanup
 INFO Running self test for shell sh
 INFO Running self test for shell bash
 WARN SelfTest([ShellFailed { shell: Sh, command: "\"sh\" \"-lc\" \"exec nix build --option substitute false --option post-build-hook '' --no-link --expr 'derivation { name = \\\"self-test-sh-1786129379049\\\"; system = \\\"x86_64-linux\\\"; builder = \\\"/bin/sh\\\"; args = [\\\"-c\\\" \\\"echo hello > \\\\$out\\\"]; }'\"", output: Output { status: ExitStatus(unix_wait_status(256)), stdout: "", stderr: "error:\n       … while calling the 'derivationStrict' builtin\n         at «nix-internal»/derivation-internal.nix:37:12:\n           36|\n           37|   strict = 

In [3]:
# @title Get LibreLane {display-mode: "form"}
# @markdown Click the ▷ button to download and install LibreLane.
# @markdown
# @markdown This will install LibreLane's tool dependencies using Nix,
# @markdown and LibreLane itself using PIP.
# @markdown
# @markdown Note that `python3-tk` may need to be installed using your OS's
# @markdown package manager.
import os
import yaml
import subprocess
import IPython

librelane_version = "latest"  # @param {key:"LibreLane Version", type:"string"}

if librelane_version == "latest":
    librelane_version = "main"

pdk_root = "~/.ciel"  # @param {key:"PDK Root", type:"string"}

pdk_root = os.path.expanduser(pdk_root)

pdk = "sky130"  # @param {key:"PDK (without the variant)", type:"string"}

librelane_ipynb_path = os.path.join(os.getcwd(), "librelane_ipynb")

display(IPython.display.HTML("<h3>Downloading LibreLane…</a>"))


TESTING_LOCALLY = False
!rm -rf {librelane_ipynb_path}
!mkdir -p {librelane_ipynb_path}
if TESTING_LOCALLY:
    !ln -s {os.getcwd()} {librelane_ipynb_path}
else:
    !curl -L "https://github.com/librelane/librelane/tarball/{librelane_version}" | tar -xzC {librelane_ipynb_path} --strip-components 1

try:
    import tkinter
except ImportError:
    if "google.colab" in sys.modules:
        !sudo apt-get install python-tk

try:
    import tkinter
except ImportError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to import the <code>tkinter</code> library for Python, which is required to load PDK configuration values. Make sure <code>python3-tk</code> or equivalent is installed on your system.</a>'
        )
    )
    raise e from None


display(IPython.display.HTML("<h3>Downloading LibreLane's dependencies…</a>"))
try:
    with subprocess.Popen(
        [
            "nix",
            "profile",
            "install",
            ".#colab-env",
        ],
        cwd=librelane_ipynb_path,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        encoding="utf8",
    ) as p:
        for line in p.stdout:
            print(line, end="")
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install binary dependencies using Nix…</h3>'
        )
    )

display(IPython.display.HTML("<h3>Downloading Python dependencies using PIP…</a>"))
try:
    subprocess.check_call(
        ["pip3", "install", "."],
        cwd=librelane_ipynb_path,
    )
except subprocess.CalledProcessError as e:
    display(
        IPython.display.HTML(
            '<h3 style="color: #800020";>❌ Failed to install Python dependencies using PIP…</h3>'
        )
    )
    raise e from None

display(IPython.display.HTML("<h3>Downloading PDK…</a>"))
import ciel
from ciel.source import StaticWebDataSource

with open(
    os.path.join(librelane_ipynb_path, "librelane", "pdk_hashes.yaml"), "r"
) as file:
    pdk_hashes = yaml.safe_load(file)

ciel.enable(
    ciel.get_ciel_home(pdk_root),
    pdk,
    pdk_hashes[pdk],
    data_source=StaticWebDataSource("https://fossi-foundation.github.io/ciel-releases"),
)

sys.path.insert(0, librelane_ipynb_path)
display(IPython.display.HTML("<h3>⭕️ Done.</a>"))

import logging

# Remove the stupid default colab logging handler
logging.getLogger().handlers.clear()

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 8905k    0 8905k    0     0  9905k      0 --:--:-- --:--:-- --:--:-- 17.5M


unpacking 'github:fossi-foundation/nix-eda/8f990fb77529c09e540e453cd836af9930ec58db?narHash=sha256-nSKBMGP8/ZC7qB3Lzd%2BFwM8REqOxlh8wpYDf2hlK6Gg%3D' into the Git cache...
copying path '/nix/store/sizirny50f893gx0gbivbqyys4fxghnq-source' from 'https://cache.nixos.org'...
unpacking 'github:numtide/devshell/255a2b1725a20d060f566e4755dbf571bbbb5f76?narHash=sha256-460jc0%2BCZfyaO8%2Bw8JNtlClB2n4ui1RbHfPTLkpwhU8%3D' into the Git cache...
copying path '/nix/store/7fd6kzwyq6pcaigmq887p9wccl7v3dl6-source' from 'https://cache.nixos.org'...
this derivation will be built:
  /nix/store/jgd55f2c85x2qc6jbynqpawn9ky5vxml-librelane-colab-env.drv
these 321 paths will be fetched (819.9 MiB download, 4.8 GiB unpacked):
  /nix/store/6rkc8a7wg89fvpvdnq2nvjk46wya6jly-abseil-cpp-20240722.1
  /nix/store/7qfvcajvjs89fxqk4379zhbdmlmxjaxb-abseil-cpp-20250814.1
  /nix/store/fwfpzqrvzhpjp91rbnq64z07diyicbwh-acl-2.3.2
  /nix/store/gmpw5fapb46sm5fxirwsyl1zicx2mrax-alsa-lib-1.2.14
  /nix/store/7l478kjr588ic8l6l32qxr60

Version 8afc8346a57fe1ab7934ba5a6056ea8b43078e71 not found locally, attempting to download…

Output()

Output()

...

Output()

Output()

Output()

Output()

Version 8afc8346a57fe1ab7934ba5a6056ea8b43078e71 enabled for the sky130 PDK.

In [4]:
import librelane

print(librelane.__version__)

3.0.6


### Creating the design

Now that LibreLane is set up, we can write a Verilog file as follows:

In [35]:
%%writefile kem.v
// Fully working ML-KEM CCA; LEVELS = 512 | 768 | 1024
module kem #(
    parameter LEVEL = 512
)(
    input clk,
    input reset,
    input start,
    input [1:0] mode,
    input [255:0] d_seed,
    input [255:0] z_seed,
    input [255:0] m_msg,
    input [7:0] din,
    input din_valid,
    input din_last,
    input [1:0]  din_sel,
    output reg [255:0] ss_out,
    output reg done,
    output reg decaps_ok,
    output reg [15:0] status
);

    localparam integer K = (LEVEL == 512) ? 2 : (LEVEL == 768) ? 3 : 4;
    localparam integer ETA1 = (LEVEL == 512) ? 3 : 2;
    localparam integer DU = (LEVEL == 1024) ? 11 : 10;
    localparam integer DV = (LEVEL == 1024) ? 5 : 4;
    localparam integer PK_LEN = 384 * K + 32;
    localparam integer SK_PKE = 384 * K;
    localparam integer SK_LEN = SK_PKE + PK_LEN + 64;
    localparam integer CT_LEN = 32 * DU * K + 32 * DV;
    localparam integer U_BYTES = 32 * DU;

    localparam [11:0] q = 12'd3329;
    localparam [3:0] S0 = 4'd0;
    localparam [3:0] E0 = 4'd4;
    localparam [3:0] T0 = 4'd8;
    localparam [3:0] SA = 4'd12;
    localparam [3:0] SW = 4'd13;
    localparam [3:0] SV = 4'd14;
    localparam [3:0] SM = 4'd15;
    localparam [3:0] OP_CLR = 4'd0;
    localparam [3:0] OP_TOM = 4'd1;
    localparam [3:0] OP_FRM = 4'd2;
    localparam [3:0] OP_NTT = 4'd3;
    localparam [3:0] OP_INT = 4'd4;
    localparam [3:0] OP_ADD = 4'd5;
    localparam [3:0] OP_SUB = 4'd6;
    localparam [3:0] OP_BMU = 4'd7;

    reg [11:0] P [0:4095];
    reg [7:0] pk_mem [0:2047];
    reg [7:0] sk_mem [0:4095];
    reg [7:0] ct_mem [0:2047];
    reg [7:0] ct2_mem [0:2047];
    reg [255:0] rho, sigma, K_bar, h_pk, z_r, m_r;
    reg [1:0] mode_r;
    reg reenc;
    reg ops_start;
    reg ops_kick;
    reg [3:0] ops_cmd, ops_sa, ops_sb, ops_sc;
    reg cpu_we;
    reg [12:0] cpu_addr;
    reg [11:0] cpu_wdata;
    reg sn_start;
    reg [7:0] sn_i, sn_j;
    reg c2s, c3s;
    reg [7:0] cnonce;
    reg gs, hs, js;
    reg [7:0] gdin, hdin, jdin;
    reg gdv, gdl, gsq, hdv, hdl, hsq, jdv, jdl, jsq;
    reg [11:0] xd, add_a, add_b;
    reg [7:0]  phase, sub;
    reg [15:0] idx, bidx, widx;
    reg [3:0] pi, pj;
    reg [31:0] bit_buf;
    reg [5:0] nbits;
    reg [7:0] hout [0:63];
    reg [15:0] hpos;
    reg [11:0] tmpv;
    reg ct_ok;

    integer ii;

    wire ops_busy, ops_done;
    wire [12:0] ops_addr;
    wire [11:0] ops_wdata;
    wire ops_we;
    wire [11:0] ops_rdata = P[ops_addr];
    wire [11:0] sn_c;
    wire sn_cv, sn_done;
    wire [11:0] c2c, c3c;
    wire c2v, c3v, c2d, c3d;
    wire grdy, hrdy, jrdy, gad, had, jad;
    wire [7:0] gdout, hdout, jdout;
    wire gdv_o, hdv_o, jdv_o;
    wire [11:0] add_r, xd1, xd10, xd11, xd4, xd5;
    wire [9:0] xc10;
    wire [10:0] xc11;
    wire [3:0] xc4;
    wire [4:0] xc5;
    wire xc1;

    kem_ops u_ops(
        .clk(clk),
        .reset(reset),
        .start(ops_start),
        .cmd(ops_cmd),
        .slot_a(ops_sa),
        .slot_b(ops_sb),
        .slot_c(ops_sc),
        .busy(ops_busy),
        .done(ops_done),
        .mem_addr(ops_addr),
        .mem_wdata(ops_wdata),
        .mem_we(ops_we),
        .mem_rdata(ops_rdata)
    );

    always @(posedge clk) begin
        if (ops_we)
            P[ops_addr] <= ops_wdata;
        else if (cpu_we)
            P[cpu_addr] <= cpu_wdata;
    end

    sample_ntt u_sn(
        .clk(clk),
        .reset(reset),
        .start(sn_start),
        .rho(rho),
        .i(sn_i),
        .j(sn_j),
        .coeff_out(sn_c),
        .coeff_valid(sn_cv),
        .done(sn_done)
    );

    sample_poly_CBD #(.ETA(2)) u_c2(
        .clk(clk),
        .reset(reset),
        .start(c2s),
        .seed(sigma),
        .nonce(cnonce),
        .coeff_out(c2c),
        .coeff_valid(c2v),
        .done(c2d)
    );

    sample_poly_CBD #(.ETA(3)) u_c3(
        .clk(clk),
        .reset(reset),
        .start(c3s),
        .seed(sigma),
        .nonce(cnonce),
        .coeff_out(c3c),
        .coeff_valid(c3v),
        .done(c3d)
    );

    sha3_512 u_g(
        .clk(clk),
        .reset(reset),
        .start(gs),
        .din(gdin),
        .din_valid(gdv),
        .din_last(gdl),
        .squeeze(gsq),
        .ready(grdy),
        .dout(gdout),
        .dout_valid(gdv_o),
        .absorb_done(gad)
    );

    sha3_256 u_h(
        .clk(clk),
        .reset(reset),
        .start(hs),
        .din(hdin),
        .din_valid(hdv),
        .din_last(hdl),
        .squeeze(hsq),
        .ready(hrdy),
        .dout(hdout),
        .dout_valid(hdv_o),
        .absorb_done(had)
    );

    shake256 u_j(
        .clk(clk),
        .reset(reset),
        .start(js),
        .din(jdin),
        .din_valid(jdv),
        .din_last(jdl),
        .squeeze(jsq),
        .ready(jrdy),
        .dout(jdout),
        .dout_valid(jdv_o),
        .absorb_done(jad)
    );

    mod_add u_add1(
        .A(add_a),
        .B(add_b),
        .result(add_r)
    );

    compress #(.D(10)) uc10(
        .x(xd),
        .compressed_x(xc10)
    );

    compress #(.D(11)) uc11(
        .x(xd),
        .compressed_x(xc11)
    );

    compress #(.D(4)) uc4(
        .x(xd),
        .compressed_x(xc4)
    );

    compress #(.D(5)) uc5(
        .x(xd),
        .compressed_x(xc5)
    );

    compress #(.D(1)) uc1(
        .x(xd),
        .compressed_x(xc1)
    );

    decompress #(.D(1)) ud1(
        .y(xd[0]),
        .decompressed_y(xd1)
    );

    decompress #(.D(10)) ud10(
        .y(xd[9:0]),
        .decompressed_y(xd10)
    );

    decompress #(.D(11)) ud11(
        .y(xd[10:0]),
        .decompressed_y(xd11)
    );

    decompress #(.D(4)) ud4(
        .y(xd[3:0]),
        .decompressed_y(xd4)
    );

    decompress #(.D(5)) ud5(
        .y(xd[4:0]),
        .decompressed_y(xd5)
    );

    always @(posedge clk or posedge reset) begin
        if (reset)
            bidx <= 16'd0;
        else if (din_valid && (phase == 8'd0)) begin
            if (din_sel == 2'd0)
                pk_mem[bidx] <= din;
            else if (din_sel == 2'd1)
                sk_mem[bidx] <= din;
            else
                ct_mem[bidx] <= din;
            if (din_last)
                bidx <= 16'd0;
            else
                bidx <= bidx + 16'd1;
        end
    end

    always @(posedge clk or posedge reset) begin
        if (reset) begin
            phase <= 8'd0;
            sub <= 8'd0;
            done <= 1'b0;
            decaps_ok <= 1'b0;
            ss_out <= 256'd0;
            status <= 16'h0001;
            ops_start <= 1'b0;
            ops_kick <= 1'b0;
            sn_start <= 1'b0;
            c2s <= 1'b0;
            c3s <= 1'b0;
            gs <= 1'b0;
            hs <= 1'b0;
            js <= 1'b0;
            gdv <= 1'b0;
            gdl <= 1'b0;
            gsq <= 1'b0;
            hdv <= 1'b0;
            hdl <= 1'b0;
            hsq <= 1'b0;
            jdv <= 1'b0;
            jdl <= 1'b0;
            jsq <= 1'b0;
            cpu_we <= 1'b0;
            idx <= 16'd0;
            pi <= 4'd0;
            pj <= 4'd0;
            hpos <= 16'd0;
            bit_buf <= 32'd0;
            nbits <= 6'd0;
            xd <= 12'd0;
            add_a <= 12'd0;
            add_b <= 12'd0;
            widx <= 16'd0;
            ct_ok <= 1'b0;
            reenc <= 1'b0;
            rho <= 256'd0;
            sigma <= 256'd0;
            K_bar <= 256'd0;
            h_pk <= 256'd0;
            z_r <= 256'd0;
            m_r <= 256'd0;
            mode_r <= 2'd0;
        end
        else begin
            ops_start <= 1'b0;
            if (ops_kick)
                ops_start <= 1'b1;

            ops_kick <= 1'b0;
            sn_start <= 1'b0;
            c2s <= 1'b0;
            c3s <= 1'b0;
            gs <= 1'b0;
            hs <= 1'b0;
            js <= 1'b0;
            gdv <= 1'b0;
            gdl <= 1'b0;
            gsq <= 1'b0;
            hdv <= 1'b0;
            hdl <= 1'b0;
            hsq <= 1'b0;
            jdv <= 1'b0;
            jdl <= 1'b0;
            jsq <= 1'b0;
            cpu_we <= 1'b0;
            done <= 1'b0;

            case (phase)
            8'd0: begin // IDLE
                if (start) begin
                    mode_r <= mode;
                    z_r <= z_seed;
                    m_r <= m_msg;
                    decaps_ok <= 1'b0;
                    reenc <= 1'b0;
                    pi <= 4'd0;
                    pj <= 4'd0;
                    sub <= 8'd0;
                    idx <= 16'd0;
                    status <= {mode, 14'd0};
                    if (mode == 2'd0) begin
                        gs <= 1'b1;
                        hpos <= 16'd0;
                        phase <= 8'd1;
                    end
                    else if (mode == 2'd1) begin
                        hs <= 1'b1;
                        hpos <= 16'd0;
                        phase <= 8'd40;
                    end
                    else begin
                        phase <= 8'd80;
                        sub <= 8'd0;
                    end
                end
            end

            // ===================== KEYGEN =====================
            // G(d||k) -> rho, sigma (33 bytes: d_seed[0..31] + K[7:0])
            8'd1: begin
                if (sub == 8'd0) begin
                    if (grdy) begin
                        if (hpos < 16'd32)
                            gdin <= d_seed[8*hpos +: 8];
                        else
                            gdin <= K[7:0];
                        gdv <= 1'b1;
                        gdl <= (hpos == 16'd32);
                        if (hpos == 16'd32)
                            sub <= 8'd1;
                        hpos <= hpos + 16'd1;
                    end
                end
                else if (sub == 8'd1) begin
                    if (gad) begin
                        hpos <= 16'd0;
                        sub <= 8'd2;
                    end
                end
                else if (sub == 8'd2) begin
                    gsq <= 1'b1;
                    sub <= 8'd3;
                end
                else if (sub == 8'd3) begin
                    if (gdv_o) begin
                        hout[hpos] <= gdout;
                        if (hpos == 16'd63)
                            sub <= 8'd4;
                        else begin
                            hpos <= hpos + 16'd1;
                            sub <= 8'd2;
                        end
                    end
                end
                else begin
                    for (ii = 0; ii < 32; ii = ii + 1) begin
                        rho[8*ii +: 8]   <= hout[ii];
                        sigma[8*ii +: 8] <= hout[32 + ii];
                    end
                    pi <= 4'd0;
                    sub <= 8'd0;
                    phase <= 8'd2;
                end
            end

            // sample s[i], TOM, NTT
            8'd2: begin
                if (sub == 8'd0) begin
                    cnonce <= {4'd0, pi};
                    if (ETA1 == 3)
                        c3s <= 1'b1;
                    else
                        c2s <= 1'b1;
                    idx <= 16'd0;
                    sub <= 8'd1;
                end
                else if (sub == 8'd1) begin
                    if (ETA1 == 3) begin
                        if (c3v) begin
                            cpu_we <= 1'b1;
                            cpu_addr <= (S0 + pi) * 256 + idx[7:0];
                            cpu_wdata <= c3c;
                            idx <= idx + 16'd1;
                        end
                        if (c3d)
                            sub <= 8'd2;
                    end
                    else begin
                        if (c2v) begin
                            cpu_we <= 1'b1;
                            cpu_addr <= (S0 + pi) * 256 + idx[7:0];
                            cpu_wdata <= c2c;
                            idx <= idx + 16'd1;
                        end
                        if (c2d)
                            sub <= 8'd2;
                    end
                end
                else if (sub == 8'd2) begin
                    ops_cmd <= OP_TOM;
                    ops_sa <= S0 + pi;
                    ops_kick <= 1'b1;
                    sub <= 8'd3;
                end
                else if (sub == 8'd3) begin
                    if (ops_done) begin
                        ops_cmd <= OP_NTT;
                        ops_sa <= S0 + pi;
                        ops_kick <= 1'b1;
                        sub <= 8'd4;
                    end
                end
                else if (sub == 8'd4) begin
                    if (ops_done) begin
                        if (pi + 1 < K[3:0]) begin
                            pi <= pi + 4'd1;
                            sub <= 8'd0;
                        end
                        else begin
                            pi <= 4'd0;
                            sub <= 8'd0;
                            phase <= 8'd3;
                        end
                    end
                end
            end

            // sample e[i], TOM, NTT
            8'd3: begin
                if (sub == 8'd0) begin
                    cnonce <= K[7:0] + {4'd0, pi};
                    if (ETA1 == 3)
                        c3s <= 1'b1;
                    else
                        c2s <= 1'b1;
                    idx <= 16'd0;
                    sub <= 8'd1;
                end
                else if (sub == 8'd1) begin
                    if (ETA1 == 3) begin
                        if (c3v) begin
                            cpu_we <= 1'b1;
                            cpu_addr <= (E0 + pi) * 256 + idx[7:0];
                            cpu_wdata <= c3c;
                            idx <= idx + 16'd1;
                        end
                        if (c3d)
                            sub <= 8'd2;
                    end
                    else begin
                        if (c2v) begin
                            cpu_we <= 1'b1;
                            cpu_addr <= (E0 + pi) * 256 + idx[7:0];
                            cpu_wdata <= c2c;
                            idx <= idx + 16'd1;
                        end
                        if (c2d)
                            sub <= 8'd2;
                    end
                end
                else if (sub == 8'd2) begin
                    ops_cmd <= OP_TOM;
                    ops_sa <= E0 + pi;
                    ops_kick <= 1'b1;
                    sub <= 8'd3;
                end
                else if (sub == 8'd3) begin
                    if (ops_done) begin
                        ops_cmd <= OP_NTT;
                        ops_sa <= E0 + pi;
                        ops_kick <= 1'b1;
                        sub <= 8'd4;
                    end
                end
                else if (sub == 8'd4) begin
                    if (ops_done) begin
                        if (pi + 1 < K[3:0]) begin
                            pi <= pi + 4'd1;
                            sub <= 8'd0;
                        end
                        else begin
                            pi <= 4'd0;
                            pj <= 4'd0;
                            sub <= 8'd0;
                            phase <= 8'd4;
                        end
                    end
                end
            end

            // t = A*s + e   (A[i][j] = SampleNTT(rho||j||i) -> sn_i=j, sn_j=i)
            8'd4: begin
                if (sub == 8'd0) begin
                    ops_cmd <= OP_CLR;
                    ops_sa <= T0 + pi;
                    ops_kick <= 1'b1;
                    sub <= 8'd1;
                end
                else if (sub == 8'd1) begin
                    if (ops_done) begin
                        sn_i <= {4'd0, pj};
                        sn_j <= {4'd0, pi};
                        sn_start <= 1'b1;
                        idx <= 16'd0;
                        sub <= 8'd2;
                    end
                end
                else if (sub == 8'd2) begin
                    if (sn_cv) begin
                        cpu_we <= 1'b1;
                        cpu_addr <= SA * 256 + idx[7:0];
                        cpu_wdata <= sn_c;
                        idx <= idx + 16'd1;
                    end
                    if (sn_done)
                        sub <= 8'd3;
                end
                else if (sub == 8'd3) begin
                    ops_cmd <= OP_TOM;
                    ops_sa <= SA;
                    ops_kick <= 1'b1;
                    sub <= 8'd4;
                end
                else if (sub == 8'd4) begin
                    if (ops_done) begin
                        ops_cmd <= OP_BMU;
                        ops_sa <= SA;
                        ops_sb <= S0 + pj;
                        ops_sc <= T0 + pi;
                        ops_kick <= 1'b1;
                        sub <= 8'd5;
                    end
                end
                else if (sub == 8'd5) begin
                    if (ops_done) begin
                        if (pj + 1 < K[3:0]) begin
                            pj <= pj + 4'd1;
                            sn_i <= {4'd0, pj + 4'd1};
                            sn_j <= {4'd0, pi};
                            sn_start <= 1'b1;
                            idx <= 16'd0;
                            sub <= 8'd2;
                        end
                        else begin
                            ops_cmd <= OP_ADD;
                            ops_sa <= T0 + pi;
                            ops_sb <= E0 + pi;
                            ops_sc <= T0 + pi;
                            ops_kick <= 1'b1;
                            sub <= 8'd6;
                        end
                    end
                end
                else if (sub == 8'd6) begin
                    if (ops_done) begin
                        if (pi + 1 < K[3:0]) begin
                            pi <= pi + 4'd1;
                            pj <= 4'd0;
                            sub <= 8'd0;
                        end
                        else begin
                            pi <= 4'd0;
                            sub <= 8'd0;
                            phase <= 8'd5;
                        end
                    end
                end
            end

            // from_mont(t) + ByteEncode_12 -> pk || rho
            8'd5: begin
                if (sub == 8'd0) begin
                    ops_cmd <= OP_FRM;
                    ops_sa <= T0 + pi;
                    ops_kick <= 1'b1;
                    sub <= 8'd1;
                end
                else if (sub == 8'd1) begin
                    if (ops_done) begin
                        idx <= 16'd0;
                        bit_buf <= 32'd0;
                        nbits <= 6'd0;
                        widx <= pi * 16'd384;
                        sub <= 8'd2;
                    end
                end
                else if (sub == 8'd2) begin
                    bit_buf <= bit_buf | ({20'd0, P[(T0 + pi) * 256 + idx[7:0]]} << nbits);
                    nbits <= nbits + 6'd12;
                    sub <= 8'd3;
                end
                else if (sub == 8'd3) begin
                    if (nbits >= 6'd8) begin
                        pk_mem[widx] <= bit_buf[7:0];
                        bit_buf <= bit_buf >> 8;
                        nbits <= nbits - 6'd8;
                        widx <= widx + 16'd1;
                    end
                    else if (idx == 16'd255) begin
                        if (pi + 1 < K[3:0]) begin
                            pi <= pi + 4'd1;
                            sub <= 8'd0;
                        end
                        else begin
                            for (ii = 0; ii < 32; ii = ii + 1)
                                pk_mem[K * 384 + ii] <= rho[8*ii +: 8];
                            pi <= 4'd0;
                            sub <= 8'd0;
                            phase <= 8'd6;
                        end
                    end
                    else begin
                        idx <= idx + 16'd1;
                        sub <= 8'd2;
                    end
                end
            end

            // from_mont(s) -> sk_pke; append pk, H(pk), z
            8'd6: begin
                if (sub == 8'd0) begin
                    ops_cmd <= OP_FRM;
                    ops_sa <= S0 + pi;
                    ops_kick <= 1'b1;
                    sub <= 8'd1;
                end
                else if (sub == 8'd1) begin
                    if (ops_done) begin
                        idx <= 16'd0;
                        bit_buf <= 32'd0;
                        nbits <= 6'd0;
                        widx <= pi * 16'd384;
                        sub <= 8'd2;
                    end
                end
                else if (sub == 8'd2) begin
                    bit_buf <= bit_buf | ({20'd0, P[(S0 + pi) * 256 + idx[7:0]]} << nbits);
                    nbits <= nbits + 6'd12;
                    sub <= 8'd3;
                end
                else if (sub == 8'd3) begin
                    if (nbits >= 6'd8) begin
                        sk_mem[widx] <= bit_buf[7:0];
                        bit_buf <= bit_buf >> 8;
                        nbits <= nbits - 6'd8;
                        widx <= widx + 16'd1;
                    end
                    else if (idx == 16'd255) begin
                        if (pi + 1 < K[3:0]) begin
                            pi <= pi + 4'd1;
                            sub <= 8'd0;
                        end
                        else begin
                            idx <= 16'd0;
                            sub <= 8'd4;
                        end
                    end
                    else begin
                        idx <= idx + 16'd1;
                        sub <= 8'd2;
                    end
                end
                else if (sub == 8'd4) begin
                    sk_mem[SK_PKE + idx] <= pk_mem[idx];
                    if (idx + 1 == PK_LEN[15:0]) begin
                        hs <= 1'b1;
                        hpos <= 16'd0;
                        sub <= 8'd5;
                    end
                    else
                        idx <= idx + 16'd1;
                end
                else if (sub == 8'd5) begin
                    if (hrdy) begin
                        hdin <= pk_mem[hpos];
                        hdv <= 1'b1;
                        hdl <= (hpos + 1 == PK_LEN[15:0]);
                        sub <= 8'd15;
                    end
                end
                else if (sub == 8'd15) begin
                    if (hpos + 1 == PK_LEN[15:0])
                        sub <= 8'd6;
                    else begin
                        hpos <= hpos + 16'd1;
                        sub <= 8'd5;
                    end
                end
                else if (sub == 8'd6) begin
                    if (had) begin
                        hpos <= 16'd0;
                        sub <= 8'd7;
                    end
                end
                else if (sub == 8'd7) begin
                    hsq <= 1'b1;
                    sub <= 8'd8;
                end
                else if (sub == 8'd8) begin
                    if (hdv_o) begin
                        sk_mem[SK_PKE + PK_LEN + hpos] <= hdout;
                        h_pk[8*hpos +: 8] <= hdout;
                        if (hpos == 16'd31)
                            sub <= 8'd9;
                        else begin
                            hpos <= hpos + 16'd1;
                            sub <= 8'd7;
                        end
                    end
                end
                else if (sub == 8'd9) begin
                    for (ii = 0; ii < 32; ii = ii + 1)
                        sk_mem[SK_PKE + PK_LEN + 32 + ii] <= z_r[8*ii +: 8];
                    done <= 1'b1;
                    status <= 16'h0A01;
                    phase <= 8'd0;
                end
            end

            // ===================== ENCAPS =====================
            // H(pk), G(m||H) -> K_bar, r(=sigma); load rho from pk
            8'd40: begin
                if (sub == 8'd0) begin
                    if (hrdy) begin
                        hdin <= pk_mem[hpos];
                        hdv <= 1'b1;
                        hdl <= (hpos + 1 == PK_LEN[15:0]);
                        sub <= 8'd10;
                    end
                end
                else if (sub == 8'd10) begin
                    if (hpos + 1 == PK_LEN[15:0])
                        sub <= 8'd1;
                    else begin
                        hpos <= hpos + 16'd1;
                        sub <= 8'd0;
                    end
                end
                else if (sub == 8'd1) begin
                    if (had) begin
                        hpos <= 16'd0;
                        sub <= 8'd2;
                    end
                end
                else if (sub == 8'd2) begin
                    hsq <= 1'b1;
                    sub <= 8'd3;
                end
                else if (sub == 8'd3) begin
                    if (hdv_o) begin
                        h_pk[8*hpos +: 8] <= hdout;
                        if (hpos == 16'd31) begin
                            gs <= 1'b1;
                            hpos <= 16'd0;
                            sub <= 8'd4;
                        end
                        else begin
                            hpos <= hpos + 16'd1;
                            sub <= 8'd2;
                        end
                    end
                end
                else if (sub == 8'd4) begin
                    if (grdy) begin
                        if (hpos < 16'd32)
                            gdin <= m_r[8*hpos +: 8];
                        else
                            gdin <= h_pk[8*(hpos - 32) +: 8];
                        gdv <= 1'b1;
                        gdl <= (hpos == 16'd63);
                        if (hpos == 16'd63)
                            sub <= 8'd5;
                        hpos <= hpos + 16'd1;
                    end
                end
                else if (sub == 8'd5) begin
                    if (gad) begin
                        hpos <= 16'd0;
                        sub <= 8'd6;
                    end
                end
                else if (sub == 8'd6) begin
                    gsq <= 1'b1;
                    sub <= 8'd7;
                end
                else if (sub == 8'd7) begin
                    if (gdv_o) begin
                        hout[hpos] <= gdout;
                        if (hpos == 16'd63)
                            sub <= 8'd8;
                        else begin
                            hpos <= hpos + 16'd1;
                            sub <= 8'd6;
                        end
                    end
                end
                else begin
                    for (ii = 0; ii < 32; ii = ii + 1) begin
                        K_bar[8*ii +: 8] <= hout[ii];
                        sigma[8*ii +: 8] <= hout[32 + ii];
                        rho[8*ii +: 8]   <= pk_mem[K * 384 + ii];
                    end
                    pi <= 4'd0;
                    sub <= 8'd0;
                    phase <= 8'd41;
                end
            end

            // decode t + TOM
            8'd41: begin
                if (sub == 8'd0) begin
                    bidx <= pi * 16'd384;
                    idx <= 16'd0;
                    bit_buf <= 32'd0;
                    nbits <= 6'd0;
                    sub <= 8'd1;
                end
                else if (sub == 8'd1) begin
                    if (nbits < 6'd12) begin
                        bit_buf <= bit_buf | ({24'd0, pk_mem[bidx]} << nbits);
                        nbits <= nbits + 6'd8;
                        bidx <= bidx + 16'd1;
                    end
                    else begin
                        cpu_we <= 1'b1;
                        cpu_addr <= (T0 + pi) * 256 + idx[7:0];
                        cpu_wdata <= bit_buf[11:0];
                        bit_buf <= bit_buf >> 12;
                        nbits <= nbits - 6'd12;
                        if (idx == 16'd255)
                            sub <= 8'd2;
                        else
                            idx <= idx + 16'd1;
                    end
                end
                else if (sub == 8'd2) begin
                    ops_cmd <= OP_TOM;
                    ops_sa <= T0 + pi;
                    ops_kick <= 1'b1;
                    sub <= 8'd3;
                end
                else if (sub == 8'd3) begin
                    if (ops_done) begin
                        if (pi + 1 < K[3:0]) begin
                            pi <= pi + 4'd1;
                            sub <= 8'd0;
                        end
                        else begin
                            pi <= 4'd0;
                            sub <= 8'd0;
                            phase <= 8'd42;
                        end
                    end
                end
            end

            // sample r, TOM, NTT
            8'd42: begin
                if (sub == 8'd0) begin
                    cnonce <= {4'd0, pi};
                    if (ETA1 == 3)
                        c3s <= 1'b1;
                    else
                        c2s <= 1'b1;
                    idx <= 16'd0;
                    sub <= 8'd1;
                end
                else if (sub == 8'd1) begin
                    if (ETA1 == 3) begin
                        if (c3v) begin
                            cpu_we <= 1'b1;
                            cpu_addr <= (S0 + pi) * 256 + idx[7:0];
                            cpu_wdata <= c3c;
                            idx <= idx + 16'd1;
                        end
                        if (c3d)
                            sub <= 8'd2;
                    end
                    else begin
                        if (c2v) begin
                            cpu_we <= 1'b1;
                            cpu_addr <= (S0 + pi) * 256 + idx[7:0];
                            cpu_wdata <= c2c;
                            idx <= idx + 16'd1;
                        end
                        if (c2d)
                            sub <= 8'd2;
                    end
                end
                else if (sub == 8'd2) begin
                    ops_cmd <= OP_TOM;
                    ops_sa <= S0 + pi;
                    ops_kick <= 1'b1;
                    sub <= 8'd3;
                end
                else if (sub == 8'd3) begin
                    if (ops_done) begin
                        ops_cmd <= OP_NTT;
                        ops_sa <= S0 + pi;
                        ops_kick <= 1'b1;
                        sub <= 8'd4;
                    end
                end
                else if (sub == 8'd4) begin
                    if (ops_done) begin
                        if (pi + 1 < K[3:0]) begin
                            pi <= pi + 4'd1;
                            sub <= 8'd0;
                        end
                        else begin
                            pi <= 4'd0;
                            sub <= 8'd0;
                            phase <= 8'd43;
                        end
                    end
                end
            end

            // sample e1[i] and e2 (eta2=2)
            8'd43: begin
                if (sub == 8'd0) begin
                    cnonce <= K[7:0] + {4'd0, pi};
                    c2s <= 1'b1;
                    idx <= 16'd0;
                    sub <= 8'd1;
                end
                else if (sub == 8'd1) begin
                    if (c2v) begin
                        cpu_we <= 1'b1;
                        cpu_addr <= (E0 + pi) * 256 + idx[7:0];
                        cpu_wdata <= c2c;
                        idx <= idx + 16'd1;
                    end
                    if (c2d) begin
                        if (pi + 1 < K[3:0]) begin
                            pi <= pi + 4'd1;
                            sub <= 8'd0;
                        end
                        else begin
                            cnonce <= (K << 1);
                            c2s <= 1'b1;
                            idx <= 16'd0;
                            sub <= 8'd2;
                        end
                    end
                end
                else if (sub == 8'd2) begin
                    if (c2v) begin
                        cpu_we <= 1'b1;
                        cpu_addr <= SM * 256 + idx[7:0];
                        cpu_wdata <= c2c;
                        idx <= idx + 16'd1;
                    end
                    if (c2d) begin
                        pi <= 4'd0;
                        pj <= 4'd0;
                        sub <= 8'd0;
                        phase <= 8'd44;
                    end
                end
            end

            // u = INTT(A^T * r) + e1
            // At[i][j]=A[j][i] => SampleNTT(rho||i||j) => sn_i=i, sn_j=j
            8'd44: begin
                if (sub == 8'd0) begin
                    ops_cmd <= OP_CLR;
                    ops_sa <= SW;
                    ops_kick <= 1'b1;
                    sub <= 8'd1;
                end
                else if (sub == 8'd1) begin
                    if (ops_done) begin
                        sn_i <= {4'd0, pi};
                        sn_j <= {4'd0, pj};
                        sn_start <= 1'b1;
                        idx <= 16'd0;
                        sub <= 8'd2;
                    end
                end
                else if (sub == 8'd2) begin
                    if (sn_cv) begin
                        cpu_we <= 1'b1;
                        cpu_addr <= SA * 256 + idx[7:0];
                        cpu_wdata <= sn_c;
                        idx <= idx + 16'd1;
                    end
                    if (sn_done)
                        sub <= 8'd3;
                end
                else if (sub == 8'd3) begin
                    ops_cmd <= OP_TOM;
                    ops_sa <= SA;
                    ops_kick <= 1'b1;
                    sub <= 8'd4;
                end
                else if (sub == 8'd4) begin
                    if (ops_done) begin
                        ops_cmd <= OP_BMU;
                        ops_sa <= SA;
                        ops_sb <= S0 + pj;
                        ops_sc <= SW;
                        ops_kick <= 1'b1;
                        sub <= 8'd5;
                    end
                end
                else if (sub == 8'd5) begin
                    if (ops_done) begin
                        if (pj + 1 < K[3:0]) begin
                            pj <= pj + 4'd1;
                            sn_i <= {4'd0, pi};
                            sn_j <= {4'd0, pj + 4'd1};
                            sn_start <= 1'b1;
                            idx <= 16'd0;
                            sub <= 8'd2;
                        end
                        else begin
                            ops_cmd <= OP_INT;
                            ops_sa <= SW;
                            ops_kick <= 1'b1;
                            sub <= 8'd6;
                        end
                    end
                end
                else if (sub == 8'd6) begin
                    if (ops_done) begin
                        ops_cmd <= OP_FRM;
                        ops_sa <= SW;
                        ops_kick <= 1'b1;
                        sub <= 8'd7;
                    end
                end
                else if (sub == 8'd7) begin
                    if (ops_done) begin
                        ops_cmd <= OP_ADD;
                        ops_sa <= SW;
                        ops_sb <= E0 + pi;
                        ops_sc <= E0 + pi;
                        ops_kick <= 1'b1;
                        sub <= 8'd8;
                    end
                end
                else if (sub == 8'd8) begin
                    if (ops_done) begin
                        if (pi + 1 < K[3:0]) begin
                            pi <= pi + 4'd1;
                            pj <= 4'd0;
                            sub <= 8'd0;
                        end
                        else begin
                            pi <= 4'd0;
                            sub <= 8'd0;
                            phase <= 8'd45;
                        end
                    end
                end
            end

            // v = INTT(t*r) + e2 + Decompress_1(m)
            8'd45: begin
                if (sub == 8'd0) begin
                    ops_cmd <= OP_CLR;
                    ops_sa <= SV;
                    ops_kick <= 1'b1;
                    sub <= 8'd1;
                end
                else if (sub == 8'd1) begin
                    if (ops_done) begin
                        ops_cmd <= OP_BMU;
                        ops_sa <= T0 + pi;
                        ops_sb <= S0 + pi;
                        ops_sc <= SV;
                        ops_kick <= 1'b1;
                        sub <= 8'd2;
                    end
                end
                else if (sub == 8'd2) begin
                    if (ops_done) begin
                        if (pi + 1 < K[3:0]) begin
                            pi <= pi + 4'd1;
                            ops_cmd <= OP_BMU;
                            ops_sa <= T0 + (pi + 4'd1);
                            ops_sb <= S0 + (pi + 4'd1);
                            ops_sc <= SV;
                            ops_kick <= 1'b1;
                            sub <= 8'd2;
                        end
                        else begin
                            ops_cmd <= OP_INT;
                            ops_sa <= SV;
                            ops_kick <= 1'b1;
                            sub <= 8'd3;
                        end
                    end
                end
                else if (sub == 8'd3) begin
                    if (ops_done) begin
                        ops_cmd <= OP_FRM;
                        ops_sa <= SV;
                        ops_kick <= 1'b1;
                        sub <= 8'd4;
                    end
                end
                else if (sub == 8'd4) begin
                    if (ops_done) begin
                        ops_cmd <= OP_ADD;
                        ops_sa <= SV;
                        ops_sb <= SM;
                        ops_sc <= SV;
                        ops_kick <= 1'b1;
                        sub <= 8'd5;
                    end
                end
                else if (sub == 8'd5) begin
                    if (ops_done) begin
                        idx <= 16'd0;
                        sub <= 8'd6;
                    end
                end
                else if (sub == 8'd6) begin
                    xd <= {11'd0, (m_r[8*idx[15:3] +: 8] >> idx[2:0]) & 1'b1};
                    sub <= 8'd7;
                end
                else if (sub == 8'd7) begin
                    add_a <= P[SV * 256 + idx[7:0]];
                    add_b <= xd1;
                    sub <= 8'd8;
                end
                else if (sub == 8'd8) begin
                    cpu_we <= 1'b1;
                    cpu_addr <= SV * 256 + idx[7:0];
                    cpu_wdata <= add_r;
                    if (idx == 16'd255) begin
                        pi <= 4'd0;
                        widx <= 16'd0;
                        idx <= 16'd0;
                        bit_buf <= 32'd0;
                        nbits <= 6'd0;
                        sub <= 8'd0;
                        phase <= 8'd46;
                    end
                    else begin
                        idx <= idx + 16'd1;
                        sub <= 8'd6;
                    end
                end
            end

            // compress-encode CT: u in E (du), v in SV (dv)
            8'd46: begin
                if (sub == 8'd0) begin
                    idx <= 16'd0;
                    bit_buf <= 32'd0;
                    nbits <= 6'd0;
                    widx <= pi * U_BYTES[15:0];
                    sub <= 8'd1;
                end
                else if (sub == 8'd1) begin
                    xd <= P[(E0 + pi) * 256 + idx[7:0]];
                    sub <= 8'd2;
                end
                else if (sub == 8'd2) begin
                    if (DU == 11)
                        bit_buf <= bit_buf | ({21'd0, xc11} << nbits);
                    else
                        bit_buf <= bit_buf | ({22'd0, xc10} << nbits);
                    nbits <= nbits + DU[5:0];
                    sub <= 8'd3;
                end
                else if (sub == 8'd3) begin
                    if (nbits >= 6'd8) begin
                        if (reenc)
                            ct2_mem[widx] <= bit_buf[7:0];
                        else
                            ct_mem[widx] <= bit_buf[7:0];
                        bit_buf <= bit_buf >> 8;
                        nbits <= nbits - 6'd8;
                        widx <= widx + 16'd1;
                    end
                    else if (idx == 16'd255) begin
                        if (pi + 1 < K[3:0]) begin
                            pi <= pi + 4'd1;
                            sub <= 8'd0;
                        end
                        else begin
                            idx <= 16'd0;
                            bit_buf <= 32'd0;
                            nbits <= 6'd0;
                            widx <= (U_BYTES * K);
                            sub <= 8'd4;
                        end
                    end
                    else begin
                        idx <= idx + 16'd1;
                        sub <= 8'd1;
                    end
                end
                else if (sub == 8'd4) begin
                    xd <= P[SV * 256 + idx[7:0]];
                    sub <= 8'd5;
                end
                else if (sub == 8'd5) begin
                    if (DV == 5)
                        bit_buf <= bit_buf | ({27'd0, xc5} << nbits);
                    else
                        bit_buf <= bit_buf | ({28'd0, xc4} << nbits);
                    nbits <= nbits + DV[5:0];
                    sub <= 8'd6;
                end
                else if (sub == 8'd6) begin
                    if (nbits >= 6'd8) begin
                        if (reenc)
                            ct2_mem[widx] <= bit_buf[7:0];
                        else
                            ct_mem[widx] <= bit_buf[7:0];
                        bit_buf <= bit_buf >> 8;
                        nbits <= nbits - 6'd8;
                        widx <= widx + 16'd1;
                    end
                    else if (idx == 16'd255) begin
                        if (reenc) begin
                            idx <= 16'd0;
                            ct_ok <= 1'b1;
                            sub <= 8'd0;
                            phase <= 8'd90;
                        end
                        else begin
                            for (ii = 0; ii < 32; ii = ii + 1)
                                ss_out[8*ii +: 8] <= K_bar[8*ii +: 8];
                            done <= 1'b1;
                            status <= 16'h0A02;
                            phase <= 8'd0;
                        end
                    end
                    else begin
                        idx <= idx + 16'd1;
                        sub <= 8'd4;
                    end
                end
            end

            // ===================== DECAPS =====================
            // copy pk from sk; load h,z,rho
            8'd80: begin
                if (sub == 8'd0) begin
                    for (ii = 0; ii < 32; ii = ii + 1) begin
                        h_pk[8*ii +: 8] <= sk_mem[SK_PKE + PK_LEN + ii];
                        z_r[8*ii +: 8]  <= sk_mem[SK_PKE + PK_LEN + 32 + ii];
                        rho[8*ii +: 8]  <= sk_mem[SK_PKE + K * 384 + ii];
                    end
                    idx <= 16'd0;
                    sub <= 8'd1;
                end
                else if (sub == 8'd1) begin
                    pk_mem[idx] <= sk_mem[SK_PKE + idx];
                    if (idx + 1 == PK_LEN[15:0]) begin
                        pi <= 4'd0;
                        sub <= 8'd0;
                        phase <= 8'd81;
                    end
                    else
                        idx <= idx + 16'd1;
                end
            end

            // decode s + TOM (stored as NTT-domain normal)
            8'd81: begin
                if (sub == 8'd0) begin
                    bidx <= pi * 16'd384;
                    idx <= 16'd0;
                    bit_buf <= 32'd0;
                    nbits <= 6'd0;
                    sub <= 8'd1;
                end
                else if (sub == 8'd1) begin
                    if (nbits < 6'd12) begin
                        bit_buf <= bit_buf | ({24'd0, sk_mem[bidx]} << nbits);
                        nbits <= nbits + 6'd8;
                        bidx <= bidx + 16'd1;
                    end
                    else begin
                        cpu_we <= 1'b1;
                        cpu_addr <= (S0 + pi) * 256 + idx[7:0];
                        cpu_wdata <= bit_buf[11:0];
                        bit_buf <= bit_buf >> 12;
                        nbits <= nbits - 6'd12;
                        if (idx == 16'd255)
                            sub <= 8'd2;
                        else
                            idx <= idx + 16'd1;
                    end
                end
                else if (sub == 8'd2) begin
                    ops_cmd <= OP_TOM;
                    ops_sa <= S0 + pi;
                    ops_kick <= 1'b1;
                    sub <= 8'd3;
                end
                else if (sub == 8'd3) begin
                    if (ops_done) begin
                        if (pi + 1 < K[3:0]) begin
                            pi <= pi + 4'd1;
                            sub <= 8'd0;
                        end
                        else begin
                            pi <= 4'd0;
                            sub <= 8'd0;
                            phase <= 8'd82;
                        end
                    end
                end
            end

            // decode/decompress u, TOM+NTT into E
            8'd82: begin
                if (sub == 8'd0) begin
                    bidx <= pi * U_BYTES[15:0];
                    idx <= 16'd0;
                    bit_buf <= 32'd0;
                    nbits <= 6'd0;
                    sub <= 8'd1;
                end
                else if (sub == 8'd1) begin
                    if (nbits < DU[5:0]) begin
                        bit_buf <= bit_buf | ({24'd0, ct_mem[bidx]} << nbits);
                        nbits <= nbits + 6'd8;
                        bidx <= bidx + 16'd1;
                    end
                    else begin
                        xd <= bit_buf[11:0];
                        bit_buf <= bit_buf >> DU;
                        nbits <= nbits - DU[5:0];
                        sub <= 8'd2;
                    end
                end
                else if (sub == 8'd2) begin
                    cpu_we <= 1'b1;
                    cpu_addr <= (E0 + pi) * 256 + idx[7:0];
                    if (DU == 11)
                        cpu_wdata <= xd11;
                    else
                        cpu_wdata <= xd10;
                    if (idx == 16'd255)
                        sub <= 8'd3;
                    else begin
                        idx <= idx + 16'd1;
                        sub <= 8'd1;
                    end
                end
                else if (sub == 8'd3) begin
                    ops_cmd <= OP_TOM;
                    ops_sa <= E0 + pi;
                    ops_kick <= 1'b1;
                    sub <= 8'd4;
                end
                else if (sub == 8'd4) begin
                    if (ops_done) begin
                        ops_cmd <= OP_NTT;
                        ops_sa <= E0 + pi;
                        ops_kick <= 1'b1;
                        sub <= 8'd5;
                    end
                end
                else if (sub == 8'd5) begin
                    if (ops_done) begin
                        if (pi + 1 < K[3:0]) begin
                            pi <= pi + 4'd1;
                            sub <= 8'd0;
                        end
                        else begin
                            idx <= 16'd0;
                            bit_buf <= 32'd0;
                            nbits <= 6'd0;
                            bidx <= (U_BYTES * K);
                            sub <= 8'd0;
                            phase <= 8'd83;
                        end
                    end
                end
            end

            // decode/decompress v into SV
            8'd83: begin
                if (sub == 8'd0) begin
                    if (nbits < DV[5:0]) begin
                        bit_buf <= bit_buf | ({24'd0, ct_mem[bidx]} << nbits);
                        nbits <= nbits + 6'd8;
                        bidx <= bidx + 16'd1;
                    end
                    else begin
                        xd <= bit_buf[11:0];
                        bit_buf <= bit_buf >> DV;
                        nbits <= nbits - DV[5:0];
                        sub <= 8'd1;
                    end
                end
                else if (sub == 8'd1) begin
                    cpu_we <= 1'b1;
                    cpu_addr <= SV * 256 + idx[7:0];
                    if (DV == 5)
                        cpu_wdata <= xd5;
                    else
                        cpu_wdata <= xd4;
                    if (idx == 16'd255) begin
                        pi <= 4'd0;
                        sub <= 8'd0;
                        phase <= 8'd84;
                    end
                    else begin
                        idx <= idx + 16'd1;
                        sub <= 8'd0;
                    end
                end
            end

            // m' = Compress_1(v - INTT(s*u))
            8'd84: begin
                if (sub == 8'd0) begin
                    ops_cmd <= OP_CLR;
                    ops_sa <= SW;
                    ops_kick <= 1'b1;
                    sub <= 8'd1;
                end
                else if (sub == 8'd1) begin
                    if (ops_done) begin
                        ops_cmd <= OP_BMU;
                        ops_sa <= S0 + pi;
                        ops_sb <= E0 + pi;
                        ops_sc <= SW;
                        ops_kick <= 1'b1;
                        sub <= 8'd2;
                    end
                end
                else if (sub == 8'd2) begin
                    if (ops_done) begin
                        if (pi + 1 < K[3:0]) begin
                            pi <= pi + 4'd1;
                            ops_cmd <= OP_BMU;
                            ops_sa <= S0 + (pi + 4'd1);
                            ops_sb <= E0 + (pi + 4'd1);
                            ops_sc <= SW;
                            ops_kick <= 1'b1;
                            sub <= 8'd2;
                        end
                        else begin
                            ops_cmd <= OP_INT;
                            ops_sa <= SW;
                            ops_kick <= 1'b1;
                            sub <= 8'd3;
                        end
                    end
                end
                else if (sub == 8'd3) begin
                    if (ops_done) begin
                        ops_cmd <= OP_FRM;
                        ops_sa <= SW;
                        ops_kick <= 1'b1;
                        sub <= 8'd4;
                    end
                end
                else if (sub == 8'd4) begin
                    if (ops_done) begin
                        ops_cmd <= OP_SUB;
                        ops_sa <= SV;
                        ops_sb <= SW;
                        ops_sc <= SW;
                        ops_kick <= 1'b1;
                        sub <= 8'd5;
                    end
                end
                else if (sub == 8'd5) begin
                    if (ops_done) begin
                        idx <= 16'd0;
                        m_r <= 256'd0;
                        sub <= 8'd6;
                    end
                end
                else if (sub == 8'd6) begin
                    xd <= P[SW * 256 + idx[7:0]];
                    sub <= 8'd7;
                end
                else if (sub == 8'd7) begin
                    m_r <= {xc1, m_r[255:1]};
                    if (idx == 16'd255) begin
                        gs <= 1'b1;
                        hpos <= 16'd0;
                        sub <= 8'd0;
                        phase <= 8'd85;
                    end
                    else begin
                        idx <= idx + 16'd1;
                        sub <= 8'd6;
                    end
                end
            end

            // G(m'||h) -> K_bar, r; then re-encrypt
            8'd85: begin
                if (sub == 8'd0) begin
                    if (grdy) begin
                        if (hpos < 16'd32)
                            gdin <= m_r[8*hpos +: 8];
                        else
                            gdin <= h_pk[8*(hpos - 32) +: 8];
                        gdv <= 1'b1;
                        gdl <= (hpos == 16'd63);
                        if (hpos == 16'd63)
                            sub <= 8'd1;
                        hpos <= hpos + 16'd1;
                    end
                end
                else if (sub == 8'd1) begin
                    if (gad) begin
                        hpos <= 16'd0;
                        sub <= 8'd2;
                    end
                end
                else if (sub == 8'd2) begin
                    gsq <= 1'b1;
                    sub <= 8'd3;
                end
                else if (sub == 8'd3) begin
                    if (gdv_o) begin
                        hout[hpos] <= gdout;
                        if (hpos == 16'd63)
                            sub <= 8'd4;
                        else begin
                            hpos <= hpos + 16'd1;
                            sub <= 8'd2;
                        end
                    end
                end
                else begin
                    for (ii = 0; ii < 32; ii = ii + 1) begin
                        K_bar[8*ii +: 8] <= hout[ii];
                        sigma[8*ii +: 8] <= hout[32 + ii];
                    end
                    reenc <= 1'b1;
                    pi <= 4'd0;
                    sub <= 8'd0;
                    phase <= 8'd41;
                end
            end

            // compare ct2 vs ct; accept K_bar or J(z||ct)
            8'd90: begin
                if (sub == 8'd0) begin
                    if (ct2_mem[idx] != ct_mem[idx])
                        ct_ok <= 1'b0;
                    if (idx + 1 == CT_LEN[15:0]) begin
                        if (ct_ok && (ct2_mem[idx] == ct_mem[idx])) begin
                            for (ii = 0; ii < 32; ii = ii + 1)
                                ss_out[8*ii +: 8] <= K_bar[8*ii +: 8];
                            decaps_ok <= 1'b1;
                            done <= 1'b1;
                            status <= 16'h0A03;
                            reenc <= 1'b0;
                            phase <= 8'd0;
                        end
                        else begin
                            js <= 1'b1;
                            hpos <= 16'd0;
                            sub <= 8'd1;
                        end
                    end
                    else
                        idx <= idx + 16'd1;
                end
                else if (sub == 8'd1) begin
                    if (jrdy) begin
                        if (hpos < 16'd32)
                            jdin <= z_r[8*hpos +: 8];
                        else
                            jdin <= ct_mem[hpos - 16'd32];
                        jdv <= 1'b1;
                        jdl <= (hpos + 1 == (32 + CT_LEN));
                        sub <= 8'd5;
                    end
                end
                else if (sub == 8'd5) begin
                    if (hpos + 1 == (32 + CT_LEN))
                        sub <= 8'd2;
                    else begin
                        hpos <= hpos + 16'd1;
                        sub <= 8'd1;
                    end
                end
                else if (sub == 8'd2) begin
                    if (jad) begin
                        hpos <= 16'd0;
                        sub <= 8'd3;
                    end
                end
                else if (sub == 8'd3) begin
                    jsq <= 1'b1;
                    sub <= 8'd4;
                end
                else if (sub == 8'd4) begin
                    if (jdv_o) begin
                        ss_out[8*hpos +: 8] <= jdout;
                        if (hpos == 16'd31) begin
                            decaps_ok <= 1'b0;
                            done <= 1'b1;
                            status <= 16'h0A04;
                            reenc <= 1'b0;
                            phase <= 8'd0;
                        end
                        else begin
                            hpos <= hpos + 16'd1;
                            sub <= 8'd3;
                        end
                    end
                end
            end

            default: begin
                phase <= 8'd0;
            end
            endcase
        end
    end
endmodule

// Shared polynomial memory ops for ML-KEM
module kem_ops(
    input clk,
    input reset,
    input start,
    input [3:0] cmd,
    input [3:0] slot_a,
    input [3:0] slot_b,
    input [3:0] slot_c,
    output reg busy,
    output reg done,
    output reg [12:0] mem_addr,
    output reg [11:0] mem_wdata,
    output reg mem_we,
    input [11:0] mem_rdata
    );

    localparam [3:0] CMD_CLEAR = 4'd0;
    localparam [3:0] CMD_TOM = 4'd1;
    localparam [3:0] CMD_FRM = 4'd2;
    localparam [3:0] CMD_NTT = 4'd3;
    localparam [3:0] CMD_INTT = 4'd4;
    localparam [3:0] CMD_ADD = 4'd5;
    localparam [3:0] CMD_SUB = 4'd6;
    localparam [3:0] CMD_BMUL = 4'd7;
    localparam [11:0] q = 12'd3329;
    localparam [11:0] F = 12'd512;

    reg [3:0] cmd_r, sa, sb, sc;
    reg [7:0] st;
    reg [15:0] i, j, len, start_i;
    reg [6:0] zaddr;
    reg [11:0] t0, t1, t2, t3, zeta_r, a0, a1, b0, b1, mz;
    reg [11:0] mul_a, mul_b, add_a, add_b, sub_a, sub_b, mont_a;

    wire [11:0] mul_r, add_r, sub_r, tom_r, frm_r, zdata;

    mod_mult u_mul(
        .A(mul_a),
        .B(mul_b),
        .result(mul_r)
    );

    mod_add u_add(
        .A(add_a),
        .B(add_b),
        .result(add_r)
    );

    mod_sub u_sub(
        .A(sub_a),
        .B(sub_b),
        .result(sub_r)
    );

    to_montgomery u_tom(
        .A(mont_a),
        .clk(clk),
        .A_Prime(tom_r)
    );

    from_montgomery u_frm(
        .C(mont_a),
        .clk(clk),
        .result(frm_r)
    );

    twiddle_rom u_z(
        .clk(clk),
        .addr(zaddr),
        .data(zdata)
    );

    function [12:0] PA;
        input [3:0] slot;
        input [15:0] c;

        begin
            PA = slot * 256 + c[7:0];
        end
    endfunction

    always @(posedge clk or posedge reset) begin
        if (reset) begin
            busy <= 1'b0;
            done <= 1'b0;
            mem_we <= 1'b0;
            mem_addr <= 13'd0;
            mem_wdata <= 12'd0;
            st <= 8'd0;
            zaddr <= 7'd0;
        end
        else begin
            done <= 1'b0;
            mem_we <= 1'b0;

            if (start && !busy) begin
                busy <= 1'b1;
                cmd_r <= cmd;
                sa <= slot_a;
                sb <= slot_b;
                sc <= slot_c;
                i <= 16'd0;
                st <= 8'd1;
            end
            else if (busy) begin
                case (cmd_r)
                    CMD_CLEAR: begin
                        mem_addr <= PA(sa, i);
                        mem_wdata <= 12'd0;
                        mem_we <= 1'b1;

                        if (i == 16'd255) begin
                            busy <= 1'b0;
                            done <= 1'b1;
                            st <= 8'd0;
                        end
                        else i <= i + 16'd1;
                    end

                    CMD_TOM: begin
                        if (st == 8'd1) begin
                            mem_addr <= PA(sa, i);
                            st <= 8'd2;
                        end
                        else if (st == 8'd2) begin
                            mont_a <= mem_rdata;
                            st <= 8'd3;
                        end
                        else begin
                            mem_addr <= PA(sa, i);
                            mem_wdata <= tom_r;
                            mem_we <= 1'b1;

                            if (i == 16'd255) begin
                                busy <= 1'b0;
                                done <= 1'b1;
                                st <= 8'd0;
                            end
                            else begin
                                i <= i + 16'd1;
                                st <= 8'd1;
                            end
                        end
                    end

                    CMD_FRM: begin
                        if (st == 8'd1) begin
                            mem_addr <= PA(sa, i);
                            st <= 8'd2;
                        end
                        else if (st == 8'd2) begin
                            mont_a <= mem_rdata;
                            st <= 8'd3;
                        end
                        else begin
                            mem_addr <= PA(sa, i);
                            mem_wdata <= frm_r;
                            mem_we <= 1'b1;
                            if (i == 16'd255) begin
                                busy <= 1'b0;
                                done <= 1'b1;
                                st <= 8'd0;
                            end
                            else begin
                                i <= i + 16'd1;
                                st <= 8'd1;
                            end
                        end
                    end

                    CMD_ADD: begin
                        if (st == 8'd1) begin
                            mem_addr <= PA(sa, i);
                            st <= 8'd2;
                        end
                        else if (st == 8'd2) begin
                            t0 <= mem_rdata;
                            mem_addr <= PA(sb, i);
                            st <= 8'd3;
                        end
                        else if (st == 8'd3) begin
                            add_a <= t0;
                            add_b <= mem_rdata;
                            st <= 8'd4;
                        end
                        else begin
                            mem_addr <= PA(sc, i);
                            mem_wdata <= add_r;
                            mem_we <= 1'b1;
                            if (i == 16'd255) begin
                                busy <= 1'b0;
                                done <= 1'b1;
                                st <= 8'd0;
                            end
                            else begin
                                i <= i + 16'd1;
                                st <= 8'd1;
                            end
                        end
                    end

                    CMD_SUB: begin
                        if (st == 8'd1) begin
                            mem_addr <= PA(sa, i);
                            st <= 8'd2;
                        end
                        else if (st == 8'd2) begin
                            t0 <= mem_rdata;
                            mem_addr <= PA(sb, i);
                            st <= 8'd3;
                        end
                        else if (st == 8'd3) begin
                            sub_a <= t0;
                            sub_b <= mem_rdata;
                            st <= 8'd4;
                        end
                        else begin
                            mem_addr <= PA(sc, i);
                            mem_wdata <= sub_r;
                            mem_we <= 1'b1;

                            if (i == 16'd255) begin
                                busy <= 1'b0;
                                done <= 1'b1;
                                st <= 8'd0;
                            end
                            else begin
                                i <= i + 16'd1;
                                st <= 8'd1;
                            end
                        end
                    end

                    CMD_NTT: begin
                        if (st == 8'd1) begin
                            len <= 16'd128;
                            start_i <= 16'd0;
                            zaddr <= 7'd1;
                            st <= 8'd9;
                        end
                        else if (st == 8'd9) begin
                            st <= 8'd2;
                        end
                        else if (st == 8'd2) begin
                            zeta_r <= zdata;
                            j <= start_i;
                            st <= 8'd3;
                        end
                        else if (st == 8'd3) begin
                            mem_addr <= PA(sa, j + len);
                            st <= 8'd4;
                        end
                        else if (st == 8'd4) begin
                            mul_a <= mem_rdata;
                            mul_b <= zeta_r;
                            mem_addr <= PA(sa, j);
                            st <= 8'd5;
                        end
                        else if (st == 8'd5) begin
                            t1 <= mul_r;
                            t0 <= mem_rdata;
                            st <= 8'd6;
                        end
                        else if (st == 8'd6) begin
                            add_a <= t0;
                            add_b <= t1;
                            sub_a <= t0;
                            sub_b <= t1;
                            st <= 8'd7;
                        end
                        else if (st == 8'd7) begin
                            mem_addr <= PA(sa, j);
                            mem_wdata <= add_r;
                            mem_we <= 1'b1;
                            t2 <= sub_r;
                            st <= 8'd8;
                        end
                        else if (st == 8'd8) begin
                            mem_addr <= PA(sa, j + len);
                            mem_wdata <= t2;
                            mem_we <= 1'b1;

                            if (j + 1 < start_i + len) begin
                                j <= j + 16'd1;
                                st <= 8'd3;
                            end
                            else if (start_i + (len << 1) < 16'd256) begin
                                start_i <= start_i + (len << 1);
                                zaddr <= zaddr + 7'd1;
                                st <= 8'd9;
                            end
                            else if (len > 16'd2) begin
                                len <= len >> 1;
                                start_i <= 16'd0;
                                zaddr <= zaddr + 7'd1;
                                st <= 8'd9;
                            end
                            else begin
                                busy <= 1'b0;
                                done <= 1'b1;
                                st <= 8'd0;
                            end
                        end
                    end

                    CMD_INTT: begin
                        if (st == 8'd1) begin
                            len <= 16'd2;
                            start_i <= 16'd0;
                            zaddr <= 7'd127;
                            st <= 8'd9;
                        end
                        else if (st == 8'd9) begin
                            st <= 8'd2;
                        end
                        else if (st == 8'd2) begin
                            zeta_r <= zdata;
                            j <= start_i;
                            st <= 8'd3;
                        end
                        else if (st == 8'd3) begin
                            mem_addr <= PA(sa, j);
                            st <= 8'd4;
                        end
                        else if (st == 8'd4) begin
                            t0 <= mem_rdata;
                            mem_addr <= PA(sa, j + len);
                            st <= 8'd5;
                        end
                        else if (st == 8'd5) begin
                            t1 <= mem_rdata;
                            add_a <= t0;
                            add_b <= mem_rdata;
                            sub_a <= mem_rdata;
                            sub_b <= t0;
                            st <= 8'd6;
                        end
                        else if (st == 8'd6) begin
                            mem_addr <= PA(sa, j);
                            mem_wdata <= add_r;
                            mem_we <= 1'b1;
                            mul_a <= sub_r;
                            mul_b <= zeta_r;
                            st <= 8'd7;
                        end
                        else if (st == 8'd7) begin
                            mem_addr <= PA(sa, j + len);
                            mem_wdata <= mul_r;
                            mem_we <= 1'b1;

                            if (j + 1 < start_i + len) begin
                                j <= j + 16'd1;
                                st <= 8'd3;
                            end
                            else if (start_i + (len << 1) < 16'd256) begin
                                start_i <= start_i + (len << 1);
                                zaddr <= zaddr - 7'd1;
                                st <= 8'd9;
                            end
                            else if (len < 16'd128) begin
                                len <= len << 1;
                                start_i <= 16'd0;
                                zaddr <= zaddr - 7'd1;
                                st <= 8'd9;
                            end
                            else begin
                                i <= 16'd0;
                                st <= 8'd10;
                            end
                        end
                        else if (st == 8'd10) begin
                            mem_addr <= PA(sa, i);
                            st <= 8'd11;
                        end
                        else if (st == 8'd11) begin
                            mul_a <= mem_rdata;
                            mul_b <= F;
                            st <= 8'd12;
                        end
                        else begin
                            mem_addr <= PA(sa, i);
                            mem_wdata <= mul_r;
                            mem_we <= 1'b1;
                            if (i == 16'd255) begin
                                busy <= 1'b0;
                                done <= 1'b1;
                                st <= 8'd0;
                            end
                            else begin
                                i <= i + 16'd1;
                                st <= 8'd10;
                            end
                        end
                    end

                    CMD_BMUL: begin
                        if (st == 8'd1) begin
                            i <= 16'd0;
                            zaddr <= 7'd64;
                            st <= 8'd40;
                        end
                        else if (st == 8'd40) begin
                            st <= 8'd2;
                        end
                        else if (st == 8'd2) begin
                            zeta_r <= zdata;
                            sub_a <= q;
                            sub_b <= zdata;
                            st <= 8'd3;
                        end
                        else if (st == 8'd3) begin
                            mz <= sub_r;
                            mem_addr <= PA(sa, (i << 2));
                            st <= 8'd4;
                        end
                        else if (st == 8'd4) begin
                            a0 <= mem_rdata;
                            mem_addr <= PA(sa, (i << 2) + 1);
                            st <= 8'd5;
                        end
                        else if (st == 8'd5) begin
                            a1 <= mem_rdata;
                            mem_addr <= PA(sb, (i << 2));
                            st <= 8'd6;
                        end
                        else if (st == 8'd6) begin
                            b0 <= mem_rdata;
                            mem_addr <= PA(sb, (i << 2) + 1);
                            st <= 8'd7;
                        end
                        else if (st == 8'd7) begin
                            b1 <= mem_rdata;
                            mul_a <= a0;
                            mul_b <= b0;
                            st <= 8'd8;
                        end
                        else if (st == 8'd8) begin
                            t0 <= mul_r;
                            mul_a <= a1;
                            mul_b <= b1;
                            st <= 8'd9;
                        end
                        else if (st == 8'd9) begin
                            mul_a <= mul_r;
                            mul_b <= zeta_r;
                            st <= 8'd10;
                        end
                        else if (st == 8'd10) begin
                            add_a <= t0;
                            add_b <= mul_r;
                            st <= 8'd11;
                        end
                        else if (st == 8'd11) begin
                            t2 <= add_r;
                            mul_a <= a0;
                            mul_b <= b1;
                            st <= 8'd12;
                        end
                        else if (st == 8'd12) begin
                            t0 <= mul_r;
                            mul_a <= a1;
                            mul_b <= b0;
                            st <= 8'd13;
                        end
                        else if (st == 8'd13) begin
                            add_a <= t0;
                            add_b <= mul_r;
                            st <= 8'd14;
                        end
                        else if (st == 8'd14) begin
                            t3 <= add_r;
                            mem_addr <= PA(sc, (i << 2));
                            st <= 8'd15;
                        end
                        else if (st == 8'd15) begin
                            add_a <= mem_rdata;
                            add_b <= t2;
                            st <= 8'd16;
                        end
                        else if (st == 8'd16) begin
                            mem_addr <= PA(sc, (i << 2));
                            mem_wdata <= add_r;
                            mem_we <= 1'b1;
                            st <= 8'd17;
                        end
                        else if (st == 8'd17) begin
                            mem_addr <= PA(sc, (i << 2) + 1);
                            st <= 8'd18;
                        end
                        else if (st == 8'd18) begin
                            add_a <= mem_rdata;
                            add_b <= t3;
                            st <= 8'd19;
                        end
                        else if (st == 8'd19) begin
                            mem_addr <= PA(sc, (i << 2) + 1);
                            mem_wdata <= add_r;
                            mem_we <= 1'b1;
                            st <= 8'd20;
                        end
                        else if (st == 8'd20) begin
                            mem_addr <= PA(sa, (i << 2) + 2);
                            st <= 8'd21;
                        end
                        else if (st == 8'd21) begin
                            a0 <= mem_rdata;
                            mem_addr <= PA(sa, (i << 2) + 3);
                            st <= 8'd22;
                        end
                        else if (st == 8'd22) begin
                            a1 <= mem_rdata;
                            mem_addr <= PA(sb, (i << 2) + 2);
                            st <= 8'd23;
                        end
                        else if (st == 8'd23) begin
                            b0 <= mem_rdata;
                            mem_addr <= PA(sb, (i << 2) + 3);
                            st <= 8'd24;
                        end
                        else if (st == 8'd24) begin
                            b1 <= mem_rdata;
                            mul_a <= a0;
                            mul_b <= b0;
                            st <= 8'd25;
                        end
                        else if (st == 8'd25) begin
                            t0 <= mul_r;
                            mul_a <= a1;
                            mul_b <= b1;
                            st <= 8'd26;
                        end
                        else if (st == 8'd26) begin
                            mul_a <= mul_r;
                            mul_b <= mz;
                            st <= 8'd27;
                        end
                        else if (st == 8'd27) begin
                            add_a <= t0;
                            add_b <= mul_r;
                            st <= 8'd28;
                        end
                        else if (st == 8'd28) begin
                            t2 <= add_r;
                            mul_a <= a0;
                            mul_b <= b1;
                            st <= 8'd29;
                        end
                        else if (st == 8'd29) begin
                            t0 <= mul_r;
                            mul_a <= a1;
                            mul_b <= b0;
                            st <= 8'd30;
                        end
                        else if (st == 8'd30) begin
                            add_a <= t0;
                            add_b <= mul_r;
                            st <= 8'd31;
                        end
                        else if (st == 8'd31) begin
                            t3 <= add_r;
                            mem_addr <= PA(sc, (i << 2) + 2);
                            st <= 8'd32;
                        end
                        else if (st == 8'd32) begin
                            add_a <= mem_rdata;
                            add_b <= t2;
                            st <= 8'd33;
                        end
                        else if (st == 8'd33) begin
                            mem_addr <= PA(sc, (i << 2) + 2);
                            mem_wdata <= add_r;
                            mem_we <= 1'b1;
                            st <= 8'd34;
                        end
                        else if (st == 8'd34) begin
                            mem_addr <= PA(sc, (i << 2) + 3);
                            st <= 8'd35;
                        end
                        else if (st == 8'd35) begin
                            add_a <= mem_rdata;
                            add_b <= t3;
                            st <= 8'd36;
                        end
                        else if (st == 8'd36) begin
                            mem_addr <= PA(sc, (i << 2) + 3);
                            mem_wdata <= add_r;
                            mem_we <= 1'b1;
                            if (i == 16'd63) begin
                                busy <= 1'b0;
                                done <= 1'b1;
                                st <= 8'd0;
                            end
                            else begin
                                i <= i + 16'd1;
                                zaddr <= 7'd64 + i[6:0] + 7'd1;
                                st <= 8'd40;
                            end
                        end
                    end

                    default: begin
                        busy <= 1'b0;
                        st <= 8'd0;
                    end
                endcase
            end
        end
    end
endmodule

// Modular Subtraction
// Assumptions: Inputs are already modulo reduced, e.g.  0 ≤ A < q &  0 ≤ B < q, thus making  0 ≤ result < q

module mod_sub(
    input [11:0] A,
    input [11:0] B,
    output reg [11:0] result
    );

    parameter [11:0] q = 12'd3329;

    always @(*) begin
        if (A >= B) begin
            result = A - B;
        end else begin
            result = A + q - B;
        end
    end
endmodule

// Montgomery reduction
// T will be a product from mod_mult
// Assumption: 0 <= T < qR where R is 2^16 (or 65536)
module montgomery(
    input [31:0] T,
    input clk, //Dummy clock port
    output reg [11:0] t
);

    parameter [11:0] N = 3329;
    parameter [15:0] N_Prime = 3327;

    reg [15:0] m;
    reg [35:0] temp;
    reg [24:0] reduced;

    always @(*) begin
        m = (T * N_Prime) & 16'hFFFF;
        temp = T + (m * N);
        reduced = temp >> 16;

        if (reduced >= N) begin
            t = reduced - N;
        end else begin
            t = reduced[11:0];
        end
    end
endmodule

// Turns input into Montogomery domain
module to_montgomery(
    input [11:0] A,
    input clk, //Dummy clock
    output [11:0] A_Prime
);

    parameter [11:0] R_Sq_mod = 1353; // Uses R^2 mod q constant value

    wire [31:0] product;

    assign product = A * R_Sq_mod;

    montgomery APrime (
        .clk(clk), //Instantiating dummy clock
        .T(product),
        .t(A_Prime)
    );
endmodule

// Modular Multiplication
module mod_mult(
    input [11:0] A,
    input [11:0] B,
    output [11:0] result
);

    wire [31:0] product;

    assign product = A * B;

    montgomery Result (
        .T(product),
        .t(result)
    );
endmodule


module from_montgomery(
    input [11:0] C,
    input clk, //Dummy clock
    output [11:0] result
);
    wire [31:0] extended_C;

    assign extended_C = {20'd0, C};

    montgomery Result (
        .clk(clk),
        .T(extended_C),
        .t(result)
    );
endmodule

// Modular addition
// Assumptions: Inputs are already modulo reduced, e.g.  0 ≤ A < q &  0 ≤ B < q, thus making  0 ≤ result < q

module mod_add(
    input [11:0] A,
    input [11:0] B,
    output reg [11:0] result
    );

    wire [12:0] sum;
    parameter [11:0] q = 12'd3329;

    assign sum = A + B;

    always @(*) begin
        if (sum >= q) begin
            result = sum - q;
        end else begin
            result = sum;
        end
    end
endmodule



module sky130_sram_1kbyte_1rw1r_32x256_8(
    `ifdef USE_POWER_PINS
        inout vccd1,
        inout vssd1,
    `endif

    input clk0,
    input csb0,
    input web0,
    input [3:0] wmask0,
    input [7:0] addr0,
    input [31:0] din0,
    output [31:0] dout0,
    input clk1,
    input csb1,
    input [7:0] addr1,
    output [31:0] dout1
);
endmodule

// Address generator for Kyber
module address_gen(
    input clk,
    input reset,
    input start,
    input inverse,
    output reg rd_en,
    output reg [7:0] rd_addr_a,
    output reg [7:0] rd_addr_b,
    output reg [6:0] twiddle_addr,
    output wr_en,
    output [7:0] wr_addr_a,
    output [7:0] wr_addr_b,
    output reg done
    );

    integer k;
    parameter integer PIPE_LATENCY = 4;
    localparam [1:0] IDLE = 2'd0;
    localparam [1:0] RUN = 2'd1;
    localparam [1:0] DRAIN = 2'd2;
    localparam [1:0] FIN = 2'd3;

    reg [1:0] state;
    reg [2:0] stage;
    reg [7:0] group;
    reg [7:0] j;
    reg [6:0] tw_idx;
    reg [7:0] drain_cnt;
    reg we_sr [0:PIPE_LATENCY-1];
    reg [7:0] addra_sr [0:PIPE_LATENCY-1];
    reg [7:0] addrb_sr [0:PIPE_LATENCY-1];

    wire [7:0] d = inverse ? (8'd2 << stage) : (8'd128 >> stage);
    wire [7:0] groups_per_stage = inverse ? (8'd1 << (3'd6 - stage)) : (8'd1 << stage);
    wire [3:0] grp_shift = inverse ? (4'd2 + {1'b0, stage}) : (4'd8 - {1'b0, stage});
    wire last_j = (j == d - 8'd1);
    wire last_group = (group == groups_per_stage - 8'd1);
    wire stage_last = last_j & last_group;
    wire final_stage = (stage == 3'd6);

    always @(*) begin
        rd_en = (state == RUN);
        rd_addr_a = (group << grp_shift) + j;
        rd_addr_b = rd_addr_a + d;
        twiddle_addr = tw_idx;
    end

    always @(posedge clk or posedge reset) begin
        if (reset) begin
            state <= IDLE;
            stage <= 3'd0;
            group <= 8'd0;
            j <= 8'd0;
            tw_idx <= inverse ? 7'd127 : 7'd1;
            drain_cnt <= 8'd0;
            done <= 1'b0;
        end
        else begin
            case (state)
                IDLE: begin
                    if (start) begin
                        stage <= 3'd0;
                        group <= 8'd0;
                        j <= 8'd0;
                        tw_idx <= inverse ? 7'd127 : 7'd1;
                        done <= 1'b0;
                        state <= RUN;
                    end
                end

                RUN: begin
                    if (stage_last) begin
                        drain_cnt <= PIPE_LATENCY[7:0];
                        state <= DRAIN;
                    end
                    else if (last_j) begin
                        j <= 8'd0;
                        group <= group + 8'd1;
                        tw_idx <= inverse ? (tw_idx - 7'd1) : (tw_idx + 7'd1);
                    end
                    else begin
                        j <= j + 8'd1;
                    end
                end

                DRAIN: begin
                    if (drain_cnt == 8'd0) begin
                        if (final_stage) begin
                            state <= FIN;
                        end else begin
                            stage <= stage + 3'd1;
                            group <= 8'd0;
                            j <= 8'd0;
                            tw_idx <= inverse ? (tw_idx - 7'd1) : (tw_idx + 7'd1);
                            state <= RUN;
                        end
                    end
                    else begin
                        drain_cnt <= drain_cnt - 8'd1;
                    end
                end

                FIN: begin
                    done <= 1'b1;
                    state <= IDLE;
                end

                default: state <= IDLE;
            endcase
        end
    end

    always @(posedge clk or posedge reset) begin
        if (reset) begin
            for (k = 0; k < PIPE_LATENCY; k = k + 1) begin
                we_sr[k] <= 1'b0;
                addra_sr[k] <= 8'd0;
                addrb_sr[k] <= 8'd0;
            end
        end
        else begin
            we_sr[0] <= rd_en;
            addra_sr[0] <= rd_addr_a;
            addrb_sr[0] <= rd_addr_b;

            for (k = 1; k < PIPE_LATENCY; k = k + 1) begin
                we_sr[k] <= we_sr[k-1];
                addra_sr[k] <= addra_sr[k-1];
                addrb_sr[k] <= addrb_sr[k-1];
            end
        end
    end

    assign wr_en = we_sr[PIPE_LATENCY-1];
    assign wr_addr_a = addra_sr[PIPE_LATENCY-1];
    assign wr_addr_b = addrb_sr[PIPE_LATENCY-1];
endmodule

// SRAM version
module coeff_ram_sram(
    `ifdef USE_POWER_PINS
        inout vccd1,
        inout vssd1,
    `endif

    input clk,
    input rd_en,
    input wr_en,
    input [7:0] rd_addr_a,
    input [7:0] wr_addr_a,
    input [7:0] rd_addr_b,
    input [7:0] wr_addr_b,
    input [11:0] wr_data_a,
    input [11:0] wr_data_b,
    output reg [11:0] rd_data_a,
    output reg [11:0] rd_data_b
    );

    reg read_a_in_bank0_d;

    wire read_a_in_bank0 = (^rd_addr_a) == 1'b0;
    wire write_a_in_bank0 = (^wr_addr_a) == 1'b0;
    wire [6:0] b0_raddr = read_a_in_bank0 ? rd_addr_a[7:1] : rd_addr_b[7:1];
    wire [6:0] b1_raddr = read_a_in_bank0 ? rd_addr_b[7:1] : rd_addr_a[7:1];
    wire [6:0] b0_waddr = write_a_in_bank0 ? wr_addr_a[7:1] : wr_addr_b[7:1];
    wire [11:0] b0_wdata = write_a_in_bank0 ? wr_data_a : wr_data_b;
    wire [6:0] b1_waddr = write_a_in_bank0 ? wr_addr_b[7:1] : wr_addr_a[7:1];
    wire [11:0] b1_wdata = write_a_in_bank0 ? wr_data_b : wr_data_a;
    wire [31:0] b0_dout1, b1_dout1;

    always @(posedge clk) begin
        read_a_in_bank0_d <= read_a_in_bank0;
    end

    sky130_sram_1kbyte_1rw1r_32x256_8 bank0 (
        `ifdef USE_POWER_PINS
            .vccd1 (vccd1),
            .vssd1 (vssd1),
        `endif

        .clk0(clk),
        .csb0(~wr_en),
        .web0(1'b0),
        .wmask0(4'b0011),
        .addr0({1'b0, b0_waddr}),
        .din0({20'd0, b0_wdata}),
        .dout0(),
        .clk1(clk),
        .csb1(~rd_en),
        .addr1({1'b0, b0_raddr}),
        .dout1(b0_dout1)
    );

    sky130_sram_1kbyte_1rw1r_32x256_8 bank1 (
        `ifdef USE_POWER_PINS
            .vccd1 (vccd1),
            .vssd1 (vssd1),
        `endif

        .clk0(clk),
        .csb0(~wr_en),
        .web0(1'b0),
        .wmask0(4'b0011),
        .addr0({1'b0, b1_waddr}),
        .din0({20'd0, b1_wdata}),
        .dout0(),
        .clk1(clk),
        .csb1(~rd_en),
        .addr1({1'b0, b1_raddr}),
        .dout1(b1_dout1)
    );

    always @(*) begin
        rd_data_a = read_a_in_bank0_d ? b0_dout1[11:0] : b1_dout1[11:0];
        rd_data_b = read_a_in_bank0_d ? b1_dout1[11:0] : b0_dout1[11:0];
    end
endmodule

// Storage for Kyber coefficients
module coeff_ram(
    input clk,
    input rd_en,
    input wr_en,
    input [7:0] rd_addr_a,
    input [7:0] wr_addr_a,
    input [7:0] rd_addr_b,
    input [7:0] wr_addr_b,
    input [11:0] wr_data_a,
    input [11:0] wr_data_b,
    output reg [11:0] rd_data_a,
    output reg [11:0] rd_data_b
    );

    reg [11:0] bank0 [0:127];
    reg [11:0] bank1 [0:127];
    reg [11:0] b0_q, b1_q;
    reg read_a_in_bank0_d;

    wire read_a_in_bank0 = (^rd_addr_a) == 1'b0;
    wire write_a_in_bank0 = (^wr_addr_a) == 1'b0;
    wire [6:0] b0_raddr = read_a_in_bank0 ? rd_addr_a[7:1] : rd_addr_b[7:1];
    wire [6:0] b1_raddr = read_a_in_bank0 ? rd_addr_b[7:1] : rd_addr_a[7:1];
    wire [6:0] b0_waddr = write_a_in_bank0 ? wr_addr_a[7:1] : wr_addr_b[7:1];
    wire [11:0] b0_wdata = write_a_in_bank0 ? wr_data_a : wr_data_b;
    wire [6:0] b1_waddr = write_a_in_bank0 ? wr_addr_b[7:1] : wr_addr_a[7:1];
    wire [11:0] b1_wdata = write_a_in_bank0 ? wr_data_b : wr_data_a;

    always @(posedge clk) begin
        if (wr_en) begin
            bank0[b0_waddr] <= b0_wdata;
            bank1[b1_waddr] <= b1_wdata;
        end
        if (rd_en) begin
            b0_q <= bank0[b0_raddr];
            b1_q <= bank1[b1_raddr];
        end

        read_a_in_bank0_d <= read_a_in_bank0;
    end

    always @(*) begin
        rd_data_a = read_a_in_bank0_d ? b0_q : b1_q;
        rd_data_b = read_a_in_bank0_d ? b1_q : b0_q;
    end
endmodule

// Module for reading official Kyber twiddle factors
module twiddle_rom(
    input clk,
    input [6:0] addr,
    output reg [11:0] data
    );

    reg [11:0] rom [0:127];

    initial begin
        $readmemh("src/memory/twiddle.mem", rom);
    end

    always @(posedge clk) begin
        data <= rom[addr];
    end
endmodule

// NTT-domain pointwise multiplication on degree-1 pairs (Kyber basemul)
module basemul(
    input clk,
    input [11:0] a0,
    input [11:0] a1,
    input [11:0] b0,
    input [11:0] b1,
    input [11:0] twiddle,
    output reg [11:0] r0,
    output reg [11:0] r1
);

    wire [11:0] a1_b1;
    wire [11:0] a1_b1_zeta;
    wire [11:0] a0_b0;
    wire [11:0] a0_b1;
    wire [11:0] a1_b0;
    wire [11:0] r0_c;
    wire [11:0] r1_c;

    mod_mult mult_a1_b1 (
        .A(a1),
        .B(b1),
        .result(a1_b1)
    );

    mod_mult mult_a1_b1_z (
        .A(a1_b1),
        .B(twiddle),
        .result(a1_b1_zeta)
    );

    mod_mult mult_a0_b0 (
        .A(a0),
        .B(b0),
        .result(a0_b0)
    );

    mod_add add_r0 (
        .A(a0_b0),
        .B(a1_b1_zeta),
        .result(r0_c)
    );

    mod_mult mult_a0_b1 (
        .A(a0),
        .B(b1),
        .result(a0_b1)
    );

    mod_mult mult_a1_b0 (
        .A(a1),
        .B(b0),
        .result(a1_b0)
    );

    mod_add add_r1 (
        .A(a0_b1),
        .B(a1_b0),
        .result(r1_c)
    );

    always @(posedge clk) begin
        r0 <= r0_c;
        r1 <= r1_c;
    end
endmodule

// Gentleman-Sande butterfly for inverse NTT
// a_out = a + b, b_out = twiddle * (b - a)
// Subtract happens first, then the multiply - the mirror of the CT butterfly.
// Latency from stable inputs to outputs is 2 cycles, matching butterfly.v so the same PIPE_LATENCY write-back timing work
module butterfly_gs(
    input clk,
    input [11:0] a,
    input [11:0] b,
    input [11:0] twiddle,
    output [11:0] a_out,
    output [11:0] b_out
    );

    wire [11:0] sum;
    wire [11:0] diff;

    reg [11:0] sum_reg;
    reg [11:0] diff_reg;
    reg [11:0] tw_reg;

    mod_add add (
        .A(a),
        .B(b),
        .result(sum)
    );

    mod_sub sub (
        .A(b),
        .B(a),
        .result(diff)
    );

    always @(posedge clk) begin
        sum_reg <= sum;
        diff_reg <= diff;
        tw_reg <= twiddle;
    end

    assign a_out = sum_reg;

    mod_mult product (
        .A(diff_reg),
        .B(tw_reg),
        .result(b_out)
    );
endmodule

// Butterfly operations
module butterfly(
    input clk,
    input [11:0] a,
    input [11:0] b,
    input [11:0] twiddle,
    output [11:0] a_out,
    output [11:0] b_out
    );

    wire [11:0] T;

    reg [11:0] T_reg;
    reg [11:0] A_reg;

    mod_mult product (
        .A(b),
        .B(twiddle),
        .result(T)
    );

    always @(posedge clk) begin
        A_reg <= a;
        T_reg <= T;
    end

    mod_add add (
        .A(A_reg),
        .B(T_reg),
        .result(a_out)
    );

    mod_sub sub (
        .A(A_reg),
        .B(T_reg),
        .result(b_out)
    );
endmodule

// Inverse 256-point NTT
// Address_gen runs with inverse=1 (distance 2..128, twiddles 127..1), uses Gentleman-Sande butterfly (butterfly_gs)
// After the 7 stages, every coefficient is scaled by F = 512 (= R/128) via Montgomery multiply, which undoes the 128 from the incomplete transform for this montgomery.v (R = 2^16).
module intt(
    input clk,
    input reset,
    input start,
    input load,
    input [7:0] load_addr_a,
    input [7:0] load_addr_b,
    input [11:0] load_data_a,
    input [11:0] load_data_b,
    input [7:0] read_addr,
    output [11:0] read_data,
    output reg done
    );

    localparam [11:0] F = 12'd512;
    localparam integer PIPE_LATENCY = 2;
    localparam [1:0] SC_IDLE  = 2'd0;
    localparam [1:0] SC_RUN   = 2'd1;
    localparam [1:0] SC_DRAIN = 2'd2;

    reg ram_rd_en;
    reg ram_wr_en;
    reg busy;
    reg scaling;
    reg [7:0] ram_rd_addr_a;
    reg [7:0] ram_rd_addr_b;
    reg [7:0] ram_wr_addr_a;
    reg [7:0] ram_wr_addr_b;
    reg [11:0] ram_wr_data_a;
    reg [11:0] ram_wr_data_b;
    reg [1:0] sc_state;
    reg [7:0] sc_j;
    reg [7:0] sc_drain;
    reg sc_we_sr [0:PIPE_LATENCY-1];
    reg [7:0] sc_adda_sr [0:PIPE_LATENCY-1];
    reg [7:0] sc_addb_sr [0:PIPE_LATENCY-1];
    reg [11:0] scale_a_r;
    reg [11:0] scale_b_r;

    integer k;

    wire ag_done;
    wire rd_en;
    wire wr_en;
    wire [7:0] rd_addr_a;
    wire [7:0] rd_addr_b;
    wire [7:0] wr_addr_a;
    wire [7:0] wr_addr_b;
    wire [6:0] twiddle_addr;
    wire [11:0] coeff_a;
    wire [11:0] coeff_b;
    wire [11:0] butterfly_a;
    wire [11:0] butterfly_b;
    wire [11:0] twiddle;
    wire [11:0] scale_a_w;
    wire [11:0] scale_b_w;
    wire sc_rd_en = (sc_state == SC_RUN);
    wire sc_wr_en = sc_we_sr[PIPE_LATENCY-1];
    wire [7:0] sc_rd_a = {sc_j, 1'b0};
    wire [7:0] sc_rd_b = {sc_j, 1'b1};
    wire last_sc = (sc_j == 8'd127);

    always @(posedge clk or posedge reset) begin
        if (reset) begin
            busy <= 1'b0;
            scaling <= 1'b0;
            done <= 1'b0;
            sc_state <= SC_IDLE;
            sc_j <= 8'd0;
            sc_drain <= 8'd0;
        end
        else begin
            if (start) begin
                busy <= 1'b1;
                scaling <= 1'b0;
                done <= 1'b0;
                sc_state <= SC_IDLE;
                sc_j <= 8'd0;
            end
            else if (busy && ag_done && !scaling) begin
                scaling <= 1'b1;
                sc_state <= SC_RUN;
                sc_j <= 8'd0;
            end
            else if (scaling) begin
                case (sc_state)
                    SC_RUN: begin
                        if (last_sc) begin
                            sc_drain <= PIPE_LATENCY[7:0];
                            sc_state <= SC_DRAIN;
                        end
                        else begin
                            sc_j <= sc_j + 8'd1;
                        end
                    end
                    SC_DRAIN: begin
                        if (sc_drain == 8'd0) begin
                            scaling <= 1'b0;
                            busy <= 1'b0;
                            done <= 1'b1;
                            sc_state <= SC_IDLE;
                        end
                        else begin
                            sc_drain <= sc_drain - 8'd1;
                        end
                    end
                    default: sc_state <= SC_IDLE;
                endcase
            end
        end
    end

    always @(posedge clk or posedge reset) begin
        if (reset) begin
            for (k = 0; k < PIPE_LATENCY; k = k + 1) begin
                sc_we_sr[k] <= 1'b0;
                sc_adda_sr[k] <= 8'd0;
                sc_addb_sr[k] <= 8'd0;
            end

            scale_a_r <= 12'd0;
            scale_b_r <= 12'd0;
        end
        else begin
            sc_we_sr[0] <= sc_rd_en;
            sc_adda_sr[0] <= sc_rd_a;
            sc_addb_sr[0] <= sc_rd_b;
            for (k = 1; k < PIPE_LATENCY; k = k + 1) begin
                sc_we_sr[k] <= sc_we_sr[k-1];
                sc_adda_sr[k] <= sc_adda_sr[k-1];
                sc_addb_sr[k] <= sc_addb_sr[k-1];
            end

            scale_a_r <= scale_a_w;
            scale_b_r <= scale_b_w;
        end
    end

    always @(*) begin
        if (scaling) begin
            ram_rd_en = sc_rd_en;
            ram_wr_en = sc_wr_en;
            ram_rd_addr_a = sc_rd_a;
            ram_rd_addr_b = sc_rd_b;
            ram_wr_addr_a = sc_adda_sr[PIPE_LATENCY-1];
            ram_wr_addr_b = sc_addb_sr[PIPE_LATENCY-1];
            ram_wr_data_a = scale_a_r;
            ram_wr_data_b = scale_b_r;
        end
        else if (busy) begin
            ram_rd_en = rd_en;
            ram_wr_en = wr_en;
            ram_rd_addr_a = rd_addr_a;
            ram_rd_addr_b = rd_addr_b;
            ram_wr_addr_a = wr_addr_a;
            ram_wr_addr_b = wr_addr_b;
            ram_wr_data_a = butterfly_a;
            ram_wr_data_b = butterfly_b;
        end
        else begin
            ram_rd_en = ~load;
            ram_wr_en = load;
            ram_rd_addr_a = read_addr;
            ram_rd_addr_b = read_addr ^ 8'd1;
            ram_wr_addr_a = load_addr_a;
            ram_wr_addr_b = load_addr_b;
            ram_wr_data_a = load_data_a;
            ram_wr_data_b = load_data_b;
        end
    end

    assign read_data = coeff_a;

    address_gen #(.PIPE_LATENCY(2)) addresses(
        .clk(clk),
        .reset(reset),
        .start(start),
        .inverse(1'b1),
        .rd_en(rd_en),
        .rd_addr_a(rd_addr_a),
        .rd_addr_b(rd_addr_b),
        .twiddle_addr(twiddle_addr),
        .wr_en(wr_en),
        .wr_addr_a(wr_addr_a),
        .wr_addr_b(wr_addr_b),
        .done(ag_done)
    );

    coeff_ram memory(
        .clk(clk),
        .rd_en(ram_rd_en),
        .wr_en(ram_wr_en),
        .rd_addr_a(ram_rd_addr_a),
        .wr_addr_a(ram_wr_addr_a),
        .rd_addr_b(ram_rd_addr_b),
        .wr_addr_b(ram_wr_addr_b),
        .wr_data_a(ram_wr_data_a),
        .wr_data_b(ram_wr_data_b),
        .rd_data_a(coeff_a),
        .rd_data_b(coeff_b)
    );

    twiddle_rom twiddles(
        .clk(clk),
        .addr(twiddle_addr),
        .data(twiddle)
    );

    butterfly_gs butterfly_unit(
        .clk(clk),
        .a(coeff_a),
        .b(coeff_b),
        .twiddle(twiddle),
        .a_out(butterfly_a),
        .b_out(butterfly_b)
    );

    mod_mult scale_a (
        .A(coeff_a),
        .B(F),
        .result(scale_a_w)
    );

    mod_mult scale_b (
        .A(coeff_b),
        .B(F),
        .result(scale_b_w)
    );
endmodule

// 256-point NTT
module ntt256(
    input clk,
    input reset,
    input start,
    input load,
    input [7:0]  load_addr_a,
    input [7:0]  load_addr_b,
    input [11:0] load_data_a,
    input [11:0] load_data_b,
    input [7:0]  read_addr,
    output [11:0] read_data,
    output done
);

    reg ram_rd_en;
    reg ram_wr_en;
    reg running;
    reg [7:0] ram_rd_addr_a;
    reg [7:0] ram_rd_addr_b;
    reg [7:0] ram_wr_addr_a;
    reg [7:0] ram_wr_addr_b;
    reg [11:0] ram_wr_data_a;
    reg [11:0] ram_wr_data_b;

    wire rd_en;
    wire wr_en;
    wire [7:0] rd_addr_a;
    wire [7:0] rd_addr_b;
    wire [7:0] wr_addr_a;
    wire [7:0] wr_addr_b;
    wire [6:0] twiddle_addr;
    wire [11:0] coeff_a;
    wire [11:0] coeff_b;
    wire [11:0] butterfly_a;
    wire [11:0] butterfly_b;
    wire [11:0] twiddle;

    always @(posedge clk or posedge reset) begin
        if (reset)
            running <= 1'b0;
        else if (start)
            running <= 1'b1;
        else if (done)
            running <= 1'b0;
    end

    always @(*) begin
        if (running) begin
            ram_rd_en = rd_en;
            ram_wr_en = wr_en;
            ram_rd_addr_a = rd_addr_a;
            ram_rd_addr_b = rd_addr_b;
            ram_wr_addr_a = wr_addr_a;
            ram_wr_addr_b = wr_addr_b;
            ram_wr_data_a = butterfly_a;
            ram_wr_data_b = butterfly_b;
        end
        else begin
            ram_rd_en = ~load;
            ram_wr_en = load;
            ram_rd_addr_a = read_addr;
            ram_rd_addr_b = read_addr ^ 8'd1;
            ram_wr_addr_a = load_addr_a;
            ram_wr_addr_b = load_addr_b;
            ram_wr_data_a = load_data_a;
            ram_wr_data_b = load_data_b;
        end
    end

    assign read_data = coeff_a;

    address_gen #(.PIPE_LATENCY(2)) addresses(
        .clk(clk),
        .reset(reset),
        .start(start),
        .inverse(1'b0),
        .rd_en(rd_en),
        .rd_addr_a(rd_addr_a),
        .rd_addr_b(rd_addr_b),
        .twiddle_addr(twiddle_addr),
        .wr_en(wr_en),
        .wr_addr_a(wr_addr_a),
        .wr_addr_b(wr_addr_b),
        .done(done)
    );

    coeff_ram memory(
        .clk(clk),
        .rd_en(ram_rd_en),
        .wr_en(ram_wr_en),
        .rd_addr_a(ram_rd_addr_a),
        .wr_addr_a(ram_wr_addr_a),
        .rd_addr_b(ram_rd_addr_b),
        .wr_addr_b(ram_wr_addr_b),
        .wr_data_a(ram_wr_data_a),
        .wr_data_b(ram_wr_data_b),
        .rd_data_a(coeff_a),
        .rd_data_b(coeff_b)
    );

    twiddle_rom twiddles(
        .clk(clk),
        .addr(twiddle_addr),
        .data(twiddle)
    );

    butterfly butterfly_unit(
        .clk(clk),
        .a(coeff_a),
        .b(coeff_b),
        .twiddle(twiddle),
        .a_out(butterfly_a),
        .b_out(butterfly_b)
    );
endmodule
// Keccak-f[1600] permuation for hashing
module keccack(
    input clk,
    input reset,
    input start,
    input [1599:0] state_in,
    output reg [1599:0] state_out,
    output reg done
    );

    integer k;

    reg active;
    reg [4:0] round_i;
    reg [63:0] S [0:24];
    reg [63:0] A [0:24];
    reg [63:0] B [0:24];
    reg [63:0] C0, C1, C2, C3, C4;
    reg [63:0] D0, D1, D2, D3, D4;
    reg [63:0] rc;

    function [63:0] ROL;
        input [63:0] v;
        input integer n;

        begin
            if (n == 0)
                ROL = v;
            else
                ROL = (v << n) | (v >> (64 - n));
        end
    endfunction

    function [63:0] round_const;
        input [4:0] rnd;
        begin
            case (rnd)
                5'd0: round_const = 64'h0000000000000001;
                5'd1: round_const = 64'h0000000000008082;
                5'd2: round_const = 64'h800000000000808A;
                5'd3: round_const = 64'h8000000080008000;
                5'd4: round_const = 64'h000000000000808B;
                5'd5: round_const = 64'h0000000080000001;
                5'd6: round_const = 64'h8000000080008081;
                5'd7: round_const = 64'h8000000000008009;
                5'd8: round_const = 64'h000000000000008A;
                5'd9: round_const = 64'h0000000000000088;
                5'd10: round_const = 64'h0000000080008009;
                5'd11: round_const = 64'h000000008000000A;
                5'd12: round_const = 64'h000000008000808B;
                5'd13: round_const = 64'h800000000000008B;
                5'd14: round_const = 64'h8000000000008089;
                5'd15: round_const = 64'h8000000000008003;
                5'd16: round_const = 64'h8000000000008002;
                5'd17: round_const = 64'h8000000000000080;
                5'd18: round_const = 64'h000000000000800A;
                5'd19: round_const = 64'h800000008000000A;
                5'd20: round_const = 64'h8000000080008081;
                5'd21: round_const = 64'h8000000000008080;
                5'd22: round_const = 64'h0000000080000001;
                5'd23: round_const = 64'h8000000080008008;
                default: round_const = 64'h0;
            endcase
        end
    endfunction

    always @(posedge clk or posedge reset) begin
        if (reset) begin
            active <= 1'b0;
            done <= 1'b0;
            round_i <= 5'd0;
            state_out <= 1600'd0;

            for (k = 0; k < 25; k = k + 1)
                S[k] <= 64'd0;
        end
        else if (start) begin
            active <= 1'b1;
            done <= 1'b0;
            round_i <= 5'd0;
            S[0] <= state_in[63:0];
            S[1] <= state_in[127:64];
            S[2] <= state_in[191:128];
            S[3] <= state_in[255:192];
            S[4] <= state_in[319:256];
            S[5] <= state_in[383:320];
            S[6] <= state_in[447:384];
            S[7] <= state_in[511:448];
            S[8] <= state_in[575:512];
            S[9] <= state_in[639:576];
            S[10] <= state_in[703:640];
            S[11] <= state_in[767:704];
            S[12] <= state_in[831:768];
            S[13] <= state_in[895:832];
            S[14] <= state_in[959:896];
            S[15] <= state_in[1023:960];
            S[16] <= state_in[1087:1024];
            S[17] <= state_in[1151:1088];
            S[18] <= state_in[1215:1152];
            S[19] <= state_in[1279:1216];
            S[20] <= state_in[1343:1280];
            S[21] <= state_in[1407:1344];
            S[22] <= state_in[1471:1408];
            S[23] <= state_in[1535:1472];
            S[24] <= state_in[1599:1536];
        end
        else if (active) begin
            rc = round_const(round_i);

            // Theta
            C0 = S[0] ^ S[5] ^ S[10] ^ S[15] ^ S[20];
            C1 = S[1] ^ S[6] ^ S[11] ^ S[16] ^ S[21];
            C2 = S[2] ^ S[7] ^ S[12] ^ S[17] ^ S[22];
            C3 = S[3] ^ S[8] ^ S[13] ^ S[18] ^ S[23];
            C4 = S[4] ^ S[9] ^ S[14] ^ S[19] ^ S[24];

            D0 = C4 ^ ROL(C1, 1);
            D1 = C0 ^ ROL(C2, 1);
            D2 = C1 ^ ROL(C3, 1);
            D3 = C2 ^ ROL(C4, 1);
            D4 = C3 ^ ROL(C0, 1);

            A[0] = S[0] ^ D0;
            A[1] = S[1] ^ D1;
            A[2] = S[2] ^ D2;
            A[3] = S[3] ^ D3;
            A[4] = S[4] ^ D4;
            A[5] = S[5] ^ D0;
            A[6] = S[6] ^ D1;
            A[7] = S[7] ^ D2;
            A[8] = S[8] ^ D3;
            A[9] = S[9] ^ D4;
            A[10] = S[10] ^ D0;
            A[11] = S[11] ^ D1;
            A[12] = S[12] ^ D2;
            A[13] = S[13] ^ D3;
            A[14] = S[14] ^ D4;
            A[15] = S[15] ^ D0;
            A[16] = S[16] ^ D1;
            A[17] = S[17] ^ D2;
            A[18] = S[18] ^ D3;
            A[19] = S[19] ^ D4;
            A[20] = S[20] ^ D0;
            A[21] = S[21] ^ D1;
            A[22] = S[22] ^ D2;
            A[23] = S[23] ^ D3;
            A[24] = S[24] ^ D4;

            // Rho & Pi
            B[0] = ROL(A[0], 0);
            B[10] = ROL(A[1], 1);
            B[20] = ROL(A[2], 62);
            B[5] = ROL(A[3], 28);
            B[15] = ROL(A[4], 27);
            B[16] = ROL(A[5], 36);
            B[1] = ROL(A[6], 44);
            B[11] = ROL(A[7], 6);
            B[21] = ROL(A[8], 55);
            B[6] = ROL(A[9], 20);
            B[7] = ROL(A[10], 3);
            B[17] = ROL(A[11], 10);
            B[2] = ROL(A[12], 43);
            B[12] = ROL(A[13], 25);
            B[22] = ROL(A[14], 39);
            B[23] = ROL(A[15], 41);
            B[8] = ROL(A[16], 45);
            B[18] = ROL(A[17], 15);
            B[3] = ROL(A[18], 21);
            B[13] = ROL(A[19], 8);
            B[14] = ROL(A[20], 18);
            B[24] = ROL(A[21], 2);
            B[9] = ROL(A[22], 61);
            B[19] = ROL(A[23], 56);
            B[4] = ROL(A[24], 14);

            // Chi
            A[0] = B[0] ^ ((~B[1]) & B[2]);
            A[1] = B[1] ^ ((~B[2]) & B[3]);
            A[2] = B[2] ^ ((~B[3]) & B[4]);
            A[3] = B[3] ^ ((~B[4]) & B[0]);
            A[4] = B[4] ^ ((~B[0]) & B[1]);
            A[5] = B[5] ^ ((~B[6]) & B[7]);
            A[6] = B[6] ^ ((~B[7]) & B[8]);
            A[7] = B[7] ^ ((~B[8]) & B[9]);
            A[8] = B[8] ^ ((~B[9]) & B[5]);
            A[9] = B[9] ^ ((~B[5]) & B[6]);
            A[10] = B[10] ^ ((~B[11]) & B[12]);
            A[11] = B[11] ^ ((~B[12]) & B[13]);
            A[12] = B[12] ^ ((~B[13]) & B[14]);
            A[13] = B[13] ^ ((~B[14]) & B[10]);
            A[14] = B[14] ^ ((~B[10]) & B[11]);
            A[15] = B[15] ^ ((~B[16]) & B[17]);
            A[16] = B[16] ^ ((~B[17]) & B[18]);
            A[17] = B[17] ^ ((~B[18]) & B[19]);
            A[18] = B[18] ^ ((~B[19]) & B[15]);
            A[19] = B[19] ^ ((~B[15]) & B[16]);
            A[20] = B[20] ^ ((~B[21]) & B[22]);
            A[21] = B[21] ^ ((~B[22]) & B[23]);
            A[22] = B[22] ^ ((~B[23]) & B[24]);
            A[23] = B[23] ^ ((~B[24]) & B[20]);
            A[24] = B[24] ^ ((~B[20]) & B[21]);

            // Iota
            A[0] = A[0] ^ rc;

            for (k = 0; k < 25; k = k + 1)
                S[k] <= A[k];

            if (round_i == 5'd23) begin
                active <= 1'b0;
                done <= 1'b1;
                state_out[63:0] <= A[0];
                state_out[127:64] <= A[1];
                state_out[191:128] <= A[2];
                state_out[255:192] <= A[3];
                state_out[319:256] <= A[4];
                state_out[383:320] <= A[5];
                state_out[447:384] <= A[6];
                state_out[511:448] <= A[7];
                state_out[575:512] <= A[8];
                state_out[639:576] <= A[9];
                state_out[703:640] <= A[10];
                state_out[767:704] <= A[11];
                state_out[831:768] <= A[12];
                state_out[895:832] <= A[13];
                state_out[959:896] <= A[14];
                state_out[1023:960] <= A[15];
                state_out[1087:1024] <= A[16];
                state_out[1151:1088] <= A[17];
                state_out[1215:1152] <= A[18];
                state_out[1279:1216] <= A[19];
                state_out[1343:1280] <= A[20];
                state_out[1407:1344] <= A[21];
                state_out[1471:1408] <= A[22];
                state_out[1535:1472] <= A[23];
                state_out[1599:1536] <= A[24];
            end
            else begin
                done <= 1'b0;
                round_i <= round_i + 5'd1;
            end
        end
        else begin
            done <= 1'b0;
        end
    end
endmodule

// SHA3-256 hash
module sha3_256(
    input clk,
    input reset,
    input start,
    input [7:0] din,
    input din_valid,
    input din_last,
    input squeeze,
    output reg [7:0] dout,
    output reg dout_valid,
    output absorb_done,
    output ready
    );

    localparam RATE = 136;
    localparam OUT_LEN = 32;
    localparam [2:0] IDLE = 3'd0;
    localparam [2:0] ABSORB = 3'd1;
    localparam [2:0] PAD = 3'd2;
    localparam [2:0] PERM = 3'd3;
    localparam [2:0] SQUEEZE = 3'd4;

    reg [2:0] state;
    reg [2:0] return_st;
    reg [1599:0] s;
    reg [7:0] offset;
    reg [5:0] out_cnt;
    reg k_start;

    wire k_done;
    wire [1599:0] k_out;

    assign ready = (state == ABSORB);
    assign absorb_done = (state == SQUEEZE);

    keccack perm (
        .clk(clk),
        .reset(reset),
        .start(k_start),
        .state_in(s),
        .state_out(k_out),
        .done(k_done)
    );

    always @(posedge clk or posedge reset) begin
        if (reset) begin
            state <= IDLE;
            return_st <= IDLE;
            s <= 1600'd0;
            offset <= 8'd0;
            out_cnt <= 6'd0;
            k_start <= 1'b0;
            dout <= 8'd0;
            dout_valid <= 1'b0;
        end
        else begin
            k_start <= 1'b0;
            dout_valid <= 1'b0;

            if (start && (state == IDLE || state == SQUEEZE)) begin
                s <= 1600'd0;
                offset <= 8'd0;
                out_cnt <= 6'd0;
                state <= ABSORB;
            end
            else begin
                case (state)
                    ABSORB: begin
                        if (din_valid) begin
                            s[8*offset +: 8] <= s[8*offset +: 8] ^ din;

                            if (offset == (RATE - 1)) begin
                                k_start <= 1'b1;
                                offset <= 8'd0;
                                if (din_last)
                                    return_st <= PAD;
                                else
                                    return_st <= ABSORB;
                                state <= PERM;
                            end
                            else begin
                                offset <= offset + 8'd1;
                                if (din_last)
                                    state <= PAD;
                            end
                        end
                        else if (din_last) begin
                            state <= PAD;
                        end
                    end

                    PAD: begin
                        if (offset == (RATE - 1))
                            s[8*(RATE-1) +: 8] <= s[8*(RATE-1) +: 8] ^ 8'h86;
                        else begin
                            s[8*offset +: 8] <= s[8*offset +: 8] ^ 8'h06;
                            s[8*(RATE-1) +: 8] <= s[8*(RATE-1) +: 8] ^ 8'h80;
                        end
                        k_start <= 1'b1;
                        return_st <= SQUEEZE;
                        offset <= 8'd0;
                        out_cnt <= 6'd0;
                        state <= PERM;
                    end

                    PERM: begin
                        if (k_done) begin
                            s <= k_out;
                            state <= return_st;
                        end
                    end

                    SQUEEZE: begin
                        if (squeeze) begin
                            dout <= s[8*offset +: 8];
                            dout_valid <= 1'b1;
                            offset <= offset + 8'd1;
                            out_cnt <= out_cnt + 6'd1;

                            if (out_cnt == (OUT_LEN - 1))
                                state <= IDLE;
                        end
                    end

                    default: state <= IDLE;
                endcase
            end
        end
    end
endmodule
// SHA3-512 hash
module sha3_512(
    input clk,
    input reset,
    input start,
    input [7:0] din,
    input din_valid,
    input din_last,
    input squeeze,
    output reg [7:0] dout,
    output reg dout_valid,
    output absorb_done,
    output ready
    );

    localparam RATE = 72;
    localparam OUT_LEN = 64;
    localparam [2:0] IDLE = 3'd0;
    localparam [2:0] ABSORB = 3'd1;
    localparam [2:0] PAD = 3'd2;
    localparam [2:0] PERM = 3'd3;
    localparam [2:0] SQUEEZE = 3'd4;

    reg [2:0] state;
    reg [2:0] return_st;
    reg [1599:0] s;
    reg [7:0] offset;
    reg [5:0] out_cnt;
    reg k_start;

    wire k_done;
    wire [1599:0] k_out;

    assign ready = (state == ABSORB);
    assign absorb_done = (state == SQUEEZE);

    keccack perm (
        .clk(clk),
        .reset(reset),
        .start(k_start),
        .state_in(s),
        .state_out(k_out),
        .done(k_done)
    );

    always @(posedge clk or posedge reset) begin
        if (reset) begin
            state <= IDLE;
            return_st <= IDLE;
            s <= 1600'd0;
            offset <= 8'd0;
            out_cnt <= 6'd0;
            k_start <= 1'b0;
            dout <= 8'd0;
            dout_valid <= 1'b0;
        end
        else begin
            k_start <= 1'b0;
            dout_valid <= 1'b0;

            if (start && (state == IDLE || state == SQUEEZE)) begin
                s <= 1600'd0;
                offset <= 8'd0;
                out_cnt <= 6'd0;
                state <= ABSORB;
            end
            else begin
                case (state)
                    ABSORB: begin
                        if (din_valid) begin
                            s[8*offset +: 8] <= s[8*offset +: 8] ^ din;

                            if (offset == (RATE - 1)) begin
                                k_start <= 1'b1;
                                offset <= 8'd0;
                                if (din_last)
                                    return_st <= PAD;
                                else
                                    return_st <= ABSORB;
                                state <= PERM;
                            end
                            else begin
                                offset <= offset + 8'd1;
                                if (din_last)
                                    state <= PAD;
                            end
                        end
                        else if (din_last) begin
                            state <= PAD;
                        end
                    end

                    PAD: begin
                        if (offset == (RATE - 1))
                            s[8*(RATE-1) +: 8] <= s[8*(RATE-1) +: 8] ^ 8'h86;
                        else begin
                            s[8*offset +: 8] <= s[8*offset +: 8] ^ 8'h06;
                            s[8*(RATE-1) +: 8] <= s[8*(RATE-1) +: 8] ^ 8'h80;
                        end
                        k_start <= 1'b1;
                        return_st <= SQUEEZE;
                        offset <= 8'd0;
                        out_cnt <= 6'd0;
                        state <= PERM;
                    end

                    PERM: begin
                        if (k_done) begin
                            s <= k_out;
                            state <= return_st;
                        end
                    end

                    SQUEEZE: begin
                        if (squeeze) begin
                            dout <= s[8*offset +: 8];
                            dout_valid <= 1'b1;
                            offset <= offset + 8'd1;
                            out_cnt <= out_cnt + 6'd1;

                            if (out_cnt == (OUT_LEN - 1))
                                state <= IDLE;
                        end
                    end

                    default: state <= IDLE;
                endcase
            end
        end
    end
endmodule

// SHAKE128 hash
module shake128(
    input clk,
    input reset,
    input start,
    input [7:0] din,
    input din_valid,
    input din_last,
    input squeeze,
    output reg [7:0] dout,
    output reg dout_valid,
    output absorb_done,
    output ready
    );

    localparam integer RATE = 168;
    localparam [2:0] IDLE = 3'd0;
    localparam [2:0] ABSORB = 3'd1;
    localparam [2:0] PAD = 3'd2;
    localparam [2:0] PERM = 3'd3;
    localparam [2:0] SQUEEZE = 3'd4;

    reg [2:0] state;
    reg [2:0] return_st;
    reg [1599:0] s;
    reg [7:0] offset;
    reg k_start;

    wire k_done;
    wire [1599:0] k_out;

    assign ready = (state == ABSORB);
    assign absorb_done = (state == SQUEEZE);

    keccack perm (
        .clk(clk),
        .reset(reset),
        .start(k_start),
        .state_in(s),
        .state_out(k_out),
        .done(k_done)
    );

    always @(posedge clk or posedge reset) begin
        if (reset) begin
            state <= IDLE;
            return_st <= IDLE;
            s <= 1600'd0;
            offset <= 8'd0;
            k_start <= 1'b0;
            dout <= 8'd0;
            dout_valid <= 1'b0;
        end
        else begin
            k_start <= 1'b0;
            dout_valid <= 1'b0;

            if (start && (state == IDLE || state == SQUEEZE)) begin
                s <= 1600'd0;
                offset <= 8'd0;
                state <= ABSORB;
            end
            else begin
                case (state)
                    ABSORB: begin
                        if (din_valid) begin
                            s[8*offset +: 8] <= s[8*offset +: 8] ^ din;

                            if (offset == (RATE - 1)) begin
                                k_start <= 1'b1;
                                offset <= 8'd0;

                                if (din_last)
                                    return_st <= PAD;
                                else
                                    return_st <= ABSORB;
                                state <= PERM;
                            end
                            else begin
                                offset <= offset + 8'd1;
                                if (din_last)
                                    state <= PAD;
                            end
                        end
                        else if (din_last) begin
                            state <= PAD;
                        end
                    end

                    PAD: begin
                        if (offset == (RATE - 1))
                            s[8*(RATE-1) +: 8] <= s[8*(RATE-1) +: 8] ^ 8'h9F;
                        else begin
                            s[8*offset +: 8] <= s[8*offset +: 8] ^ 8'h1F;
                            s[8*(RATE-1) +: 8] <= s[8*(RATE-1) +: 8] ^ 8'h80;
                        end

                        k_start <= 1'b1;
                        return_st <= SQUEEZE;
                        offset <= 8'd0;
                        state <= PERM;
                    end

                    PERM: begin
                        if (k_done) begin
                            s <= k_out;
                            state <= return_st;
                        end
                    end

                    SQUEEZE: begin
                        if (squeeze) begin
                            dout <= s[8*offset +: 8];
                            dout_valid <= 1'b1;

                            if (offset == (RATE - 1)) begin
                                k_start <= 1'b1;
                                offset <= 8'd0;
                                return_st <= SQUEEZE;
                                state <= PERM;
                            end
                            else begin
                                offset <= offset + 8'd1;
                            end
                        end
                    end

                    default: state <= IDLE;
                endcase
            end
        end
    end
endmodule
// SHAKE256 hash
module shake256(
    input clk,
    input reset,
    input start,
    input [7:0] din,
    input din_valid,
    input din_last,
    input squeeze,
    output reg [7:0] dout,
    output reg dout_valid,
    output absorb_done,
    output ready
    );

    localparam integer RATE = 136;
    localparam [2:0] IDLE = 3'd0;
    localparam [2:0] ABSORB = 3'd1;
    localparam [2:0] PAD = 3'd2;
    localparam [2:0] PERM = 3'd3;
    localparam [2:0] SQUEEZE = 3'd4;

    reg [2:0] state;
    reg [2:0] return_st;
    reg [1599:0] s;
    reg [7:0] offset;
    reg k_start;

    wire k_done;
    wire [1599:0] k_out;

    assign ready = (state == ABSORB);
    assign absorb_done = (state == SQUEEZE);

    keccack perm (
        .clk(clk),
        .reset(reset),
        .start(k_start),
        .state_in(s),
        .state_out(k_out),
        .done(k_done)
    );

    always @(posedge clk or posedge reset) begin
        if (reset) begin
            state <= IDLE;
            return_st <= IDLE;
            s <= 1600'd0;
            offset <= 8'd0;
            k_start <= 1'b0;
            dout <= 8'd0;
            dout_valid <= 1'b0;
        end
        else begin
            k_start <= 1'b0;
            dout_valid <= 1'b0;

            if (start && (state == IDLE || state == SQUEEZE)) begin
                s <= 1600'd0;
                offset <= 8'd0;
                state <= ABSORB;
            end
            else begin
                case (state)
                    ABSORB: begin
                        if (din_valid) begin
                            s[8*offset +: 8] <= s[8*offset +: 8] ^ din;

                            if (offset == (RATE - 1)) begin
                                k_start <= 1'b1;
                                offset <= 8'd0;

                                if (din_last)
                                    return_st <= PAD;
                                else
                                    return_st <= ABSORB;
                                state <= PERM;
                            end
                            else begin
                                offset <= offset + 8'd1;
                                if (din_last)
                                    state <= PAD;
                            end
                        end
                        else if (din_last) begin
                            state <= PAD;
                        end
                    end

                    PAD: begin
                        if (offset == (RATE - 1))
                            s[8*(RATE-1) +: 8] <= s[8*(RATE-1) +: 8] ^ 8'h9F;
                        else begin
                            s[8*offset +: 8] <= s[8*offset +: 8] ^ 8'h1F;
                            s[8*(RATE-1) +: 8] <= s[8*(RATE-1) +: 8] ^ 8'h80;
                        end

                        k_start <= 1'b1;
                        return_st <= SQUEEZE;
                        offset <= 8'd0;
                        state <= PERM;
                    end

                    PERM: begin
                        if (k_done) begin
                            s <= k_out;
                            state <= return_st;
                        end
                    end

                    SQUEEZE: begin
                        if (squeeze) begin
                            dout <= s[8*offset +: 8];
                            dout_valid <= 1'b1;

                            if (offset == (RATE - 1)) begin
                                k_start <= 1'b1;
                                offset <= 8'd0;
                                return_st <= SQUEEZE;
                                state <= PERM;
                            end
                            else begin
                                offset <= offset + 8'd1;
                            end
                        end
                    end

                    default: state <= IDLE;
                endcase
            end
        end
    end
endmodule
// SampleNTT: creates the 256 coefficients required for the public matrix A
module sample_ntt(
    input clk,
    input reset,
    input start,
    input [255:0] rho,
    input [7:0] i,
    input [7:0] j,
    output reg [11:0] coeff_out,
    output reg coeff_valid,
    output reg done
    );

    localparam [11:0] q = 12'd3329;
    localparam [2:0] IDLE = 3'd0;
    localparam [2:0] ABSORB_SEED = 3'd1;
    localparam [2:0] WAIT_XOF = 3'd2;
    localparam [2:0] REQ_BYTE = 3'd3;
    localparam [2:0] GOT_BYTE = 3'd4;
    localparam [2:0] TRY_D1 = 3'd5;
    localparam [2:0] TRY_D2 = 3'd6;
    localparam [2:0] FINISH = 3'd7;

    reg [2:0] state;
    reg [5:0] seed_idx;
    reg [1:0] byte_cnt;
    reg [7:0] b0, b1;
    reg [8:0] ncoeffs;
    reg [11:0] d1, d2;
    reg shake_start;
    reg [7:0] shake_din;
    reg shake_din_valid;
    reg shake_din_last;
    reg shake_squeeze;

    wire shake_ready;
    wire [7:0] shake_dout;
    wire shake_dout_valid;
    wire shake_absorb_done;

    shake128 xof (
        .clk(clk),
        .reset(reset),
        .start(shake_start),
        .din(shake_din),
        .din_valid(shake_din_valid),
        .din_last(shake_din_last),
        .ready(shake_ready),
        .squeeze(shake_squeeze),
        .dout(shake_dout),
        .dout_valid(shake_dout_valid),
        .absorb_done(shake_absorb_done)
    );

    always @(posedge clk or posedge reset) begin
        if (reset) begin
            state <= IDLE;
            seed_idx <= 6'd0;
            byte_cnt <= 2'd0;
            b0 <= 8'd0;
            b1 <= 8'd0;
            ncoeffs <= 9'd0;
            d1 <= 12'd0;
            d2 <= 12'd0;
            coeff_out <= 12'd0;
            coeff_valid <= 1'b0;
            done <= 1'b0;
            shake_start <= 1'b0;
            shake_din <= 8'd0;
            shake_din_valid <= 1'b0;
            shake_din_last <= 1'b0;
            shake_squeeze <= 1'b0;
        end
        else begin
            shake_start <= 1'b0;
            shake_din_valid <= 1'b0;
            shake_din_last <= 1'b0;
            shake_squeeze <= 1'b0;
            coeff_valid <= 1'b0;
            done <= 1'b0;

            case (state)
                IDLE: begin
                    if (start) begin
                        seed_idx <= 6'd0;
                        byte_cnt <= 2'd0;
                        ncoeffs <= 9'd0;
                        shake_start <= 1'b1;
                        state <= ABSORB_SEED;
                    end
                end

                ABSORB_SEED: begin
                    if (shake_ready) begin
                        shake_din_valid <= 1'b1;

                        if (seed_idx < 6'd32) begin
                            shake_din <= rho[8*seed_idx +: 8];
                            shake_din_last <= 1'b0;
                            seed_idx <= seed_idx + 6'd1;
                        end
                        else if (seed_idx == 6'd32) begin
                            shake_din <= i;
                            shake_din_last <= 1'b0;
                            seed_idx <= seed_idx + 6'd1;
                        end
                        else begin
                            shake_din <= j;
                            shake_din_last <= 1'b1;
                            state <= WAIT_XOF;
                        end
                    end
                end

                WAIT_XOF: begin
                    if (shake_absorb_done) begin
                        byte_cnt <= 2'd0;
                        state <= REQ_BYTE;
                    end
                end

                REQ_BYTE: begin
                    if (shake_absorb_done) begin
                        shake_squeeze <= 1'b1;
                        state <= GOT_BYTE;
                    end
                end

                GOT_BYTE: begin
                    if (shake_dout_valid) begin
                        if (byte_cnt == 2'd0) begin
                            b0 <= shake_dout;
                            byte_cnt <= 2'd1;
                            state <= REQ_BYTE;
                        end
                        else if (byte_cnt == 2'd1) begin
                            b1 <= shake_dout;
                            byte_cnt <= 2'd2;
                            state <= REQ_BYTE;
                        end
                        else begin
                            d1 <= {b1[3:0], b0};
                            d2 <= {shake_dout, b1[7:4]};
                            state <= TRY_D1;
                        end
                    end
                end

                TRY_D1: begin
                    if (d1 < q) begin
                        coeff_out <= d1;
                        coeff_valid <= 1'b1;
                        ncoeffs <= ncoeffs + 9'd1;

                        if (ncoeffs == 9'd255)
                            state <= FINISH;
                        else
                            state <= TRY_D2;
                    end
                    else begin
                        state <= TRY_D2;
                    end
                end

                TRY_D2: begin
                    if (d2 < q) begin
                        coeff_out <= d2;
                        coeff_valid <= 1'b1;
                        ncoeffs <= ncoeffs + 9'd1;

                        if (ncoeffs == 9'd255)
                            state <= FINISH;
                        else begin
                            byte_cnt <= 2'd0;
                            state <= REQ_BYTE;
                        end
                    end
                    else begin
                        byte_cnt <= 2'd0;
                        state <= REQ_BYTE;
                    end
                end

                FINISH: begin
                    done <= 1'b1;
                    state <= IDLE;
                end
                default: state <= IDLE;
            endcase
        end
    end
endmodule
// SamplePolyCBD_eta: Creates the error and secret polynomials
module sample_poly_CBD #(
    parameter ETA = 2
)(
    input clk,
    input reset,
    input start,
    input [255:0] seed,
    input [7:0] nonce,
    output reg [11:0] coeff_out,
    output reg coeff_valid,
    output reg done
    );

    localparam [11:0] q = 12'd3329;
    localparam integer NEED_BITS = 2 * ETA;
    localparam [2:0] IDLE = 3'd0;
    localparam [2:0] ABSORB_SEED = 3'd1;
    localparam [2:0] WAIT_XOF = 3'd2;
    localparam [2:0] REQ_BYTE = 3'd3;
    localparam [2:0] GOT_BYTE = 3'd4;
    localparam [2:0] EMIT = 3'd5;
    localparam [2:0] FINISH = 3'd6;

    reg [2:0] state;
    reg [5:0] seed_idx;
    reg [8:0] ncoeffs;
    reg [15:0] bit_buf;
    reg [4:0] bit_count;
    reg shake_start;
    reg [7:0] shake_din;
    reg shake_din_valid;
    reg shake_din_last;
    reg shake_squeeze;
    reg [3:0] x_sum;
    reg [3:0] y_sum;
    reg [11:0] diff;

    wire shake_ready;
    wire [7:0] shake_dout;
    wire shake_dout_valid;
    wire shake_absorb_done;

    integer k;

    shake256 prf (
        .clk(clk),
        .reset(reset),
        .start(shake_start),
        .din(shake_din),
        .din_valid(shake_din_valid),
        .din_last(shake_din_last),
        .ready(shake_ready),
        .squeeze(shake_squeeze),
        .dout(shake_dout),
        .dout_valid(shake_dout_valid),
        .absorb_done(shake_absorb_done)
    );

    always @(posedge clk or posedge reset) begin
        if (reset) begin
            state <= IDLE;
            seed_idx <= 6'd0;
            ncoeffs <= 9'd0;
            bit_buf <= 16'd0;
            bit_count <= 5'd0;
            coeff_out <= 12'd0;
            coeff_valid <= 1'b0;
            done <= 1'b0;
            shake_start <= 1'b0;
            shake_din <= 8'd0;
            shake_din_valid <= 1'b0;
            shake_din_last <= 1'b0;
            shake_squeeze <= 1'b0;
            x_sum <= 4'd0;
            y_sum <= 4'd0;
            diff <= 12'd0;
        end
        else begin
            shake_start <= 1'b0;
            shake_din_valid <= 1'b0;
            shake_din_last <= 1'b0;
            shake_squeeze <= 1'b0;
            coeff_valid <= 1'b0;
            done <= 1'b0;

            case (state)
                IDLE: begin
                    if (start) begin
                        seed_idx <= 6'd0;
                        ncoeffs <= 9'd0;
                        bit_buf <= 16'd0;
                        bit_count <= 5'd0;
                        shake_start <= 1'b1;
                        state <= ABSORB_SEED;
                    end
                end

                ABSORB_SEED: begin
                    if (shake_ready) begin
                        shake_din_valid <= 1'b1;

                        if (seed_idx < 6'd32) begin
                            shake_din <= seed[8*seed_idx +: 8];
                            shake_din_last <= 1'b0;
                            seed_idx <= seed_idx + 6'd1;
                        end
                        else begin
                            shake_din <= nonce;
                            shake_din_last <= 1'b1;
                            state <= WAIT_XOF;
                        end
                    end
                end

                WAIT_XOF: begin
                    if (shake_absorb_done)
                        state <= REQ_BYTE;
                end

                REQ_BYTE: begin
                    if (bit_count >= NEED_BITS[4:0])
                        state <= EMIT;
                    else if (shake_absorb_done) begin
                        shake_squeeze <= 1'b1;
                        state <= GOT_BYTE;
                    end
                end

                GOT_BYTE: begin
                    if (shake_dout_valid) begin
                        bit_buf <= bit_buf | ({8'd0, shake_dout} << bit_count);
                        bit_count <= bit_count + 5'd8;
                        state <= REQ_BYTE;
                    end
                end

                EMIT: begin
                    x_sum = 4'd0;
                    y_sum = 4'd0;

                    for (k = 0; k < ETA; k = k + 1) begin
                        x_sum = x_sum + bit_buf[k];
                        y_sum = y_sum + bit_buf[ETA + k];
                    end

                    if (x_sum >= y_sum)
                        diff = {8'd0, x_sum} - {8'd0, y_sum};
                    else
                        diff = q - ({8'd0, y_sum} - {8'd0, x_sum});

                    coeff_out <= diff;
                    coeff_valid <= 1'b1;
                    bit_buf <= bit_buf >> NEED_BITS;
                    bit_count <= bit_count - NEED_BITS[4:0];
                    ncoeffs <= ncoeffs + 9'd1;

                    if (ncoeffs == 9'd255)
                        state <= FINISH;
                    else
                        state <= REQ_BYTE;
                end

                FINISH: begin
                    done <= 1'b1;
                    state <= IDLE;
                end
                default: state <= IDLE;
            endcase
        end
    end
endmodule
// Data compression
module compress #(
    parameter D = 10
) (
    input [11:0] x,
    output [D-1:0] compressed_x
);

    localparam [11:0] q = 12'd3329;
    localparam [11:0] q_HALF = 12'd1664;

    wire [11+D:0] shifted = {x, {D{1'b0}}};
    wire [11+D:0] V = shifted + q_HALF;
    wire [11+D:0] quotient = V / q;

    assign compressed_x = quotient[D-1:0];
endmodule
// Data decompression
module decompress #(
    parameter D = 10
) (
    input [D-1:0] y,
    output [11:0] decompressed_y
);

    localparam [11:0] q = 12'd3329;

    wire [11+D:0] product = y * q;
    wire [11+D:0] sum = product + (1 << (D-1));

    assign decompressed_y = sum[11+D:D];
endmodule
// Packs coefficients
module pack #(
    parameter D = 12
)(
    input clk,
    input start,
    input [11:0] coeff_in,
    input coeff_valid,
    output ready,
    output reg [7:0] byte_out,
    output reg byte_valid,
    output reg done
    );

    localparam [1:0] WAIT = 2'd0;
    localparam [1:0] EMIT = 2'd1;
    localparam integer BUF_W = D + 8;

    reg active;
    reg [1:0] state;
    reg [BUF_W-1:0] bit_buf;
    reg [4:0] bits;
    reg [8:0] coeffs_done;

    wire [BUF_W-1:0] coeff_ext = {{(BUF_W-D){1'b0}}, coeff_in[D-1:0]};
    wire [4:0] bits_after_in  = bits + D[4:0];
    wire [4:0] bits_after_out = bits - 5'd8;

    assign ready = active && (state == WAIT);

    always @(posedge clk) begin
        if (start) begin
            active <= 1'b1;
            state <= WAIT;
            bit_buf <= {BUF_W{1'b0}};
            bits <= 5'd0;
            coeffs_done <= 9'd0;
            byte_valid <= 1'b0;
            byte_out <= 8'd0;
            done <= 1'b0;
        end
        else if (active) begin
            done <= 1'b0;

            case (state)
                WAIT: begin
                    byte_valid <= 1'b0;
                    if (coeff_valid) begin
                        bit_buf <= bit_buf | (coeff_ext << bits);
                        bits <= bits_after_in;
                        coeffs_done <= coeffs_done + 9'd1;

                        if (bits_after_in >= 5'd8)
                            state <= EMIT;
                    end
                end

                EMIT: begin
                    byte_out <= bit_buf[7:0];
                    byte_valid <= 1'b1;
                    bit_buf <= bit_buf >> 8;
                    bits <= bits_after_out;

                    if (bits_after_out >= 5'd8) begin
                        state <= EMIT;
                    end
                    else if ((coeffs_done == 9'd256) && (bits_after_out == 5'd0)) begin
                        done <= 1'b1;
                        active <= 1'b0;
                        state <= WAIT;
                    end
                    else begin
                        state <= WAIT;
                    end
                end

                default: state <= WAIT;
            endcase
        end
        else begin
            byte_valid <= 1'b0;
        end
    end
endmodule
// Unpacs data
module unpack #(
    parameter D = 12
)(
    input clk,
    input start,
    input [7:0] byte_in,
    input byte_valid,
    output ready,
    output reg [11:0] coeff_out,
    output reg coeff_valid,
    output reg done
    );

    localparam [1:0] WAIT = 2'd0;
    localparam [1:0] OUT = 2'd1;
    localparam integer BUF_W = D + 8;
    localparam integer N_BYTES = 32 * D;

    reg active;
    reg [1:0] state;
    reg [BUF_W-1:0] bit_buf;
    reg [4:0] bits;
    reg [8:0] coeffs_done;
    reg [9:0] bytes_done;

    wire [4:0] bits_after_in  = bits + 5'd8;
    wire [4:0] bits_after_out = bits - D[4:0];

    assign ready = active && (state == WAIT);

    always @(posedge clk) begin
        if (start) begin
            active <= 1'b1;
            state <= WAIT;
            bit_buf <= {BUF_W{1'b0}};
            bits <= 5'd0;
            coeffs_done <= 9'd0;
            bytes_done <= 10'd0;
            coeff_valid <= 1'b0;
            coeff_out <= 12'd0;
            done <= 1'b0;
        end
        else if (active) begin
            done <= 1'b0;

            case (state)
                WAIT: begin
                    coeff_valid <= 1'b0;
                    if (byte_valid) begin
                        bit_buf <= bit_buf | ({{(BUF_W-8){1'b0}}, byte_in} << bits);
                        bits <= bits_after_in;
                        bytes_done <= bytes_done + 10'd1;

                        if (bits_after_in >= D[4:0])
                            state <= OUT;
                    end
                end

                OUT: begin
                    coeff_out <= 12'd0 | bit_buf[D-1:0];
                    coeff_valid <= 1'b1;
                    bit_buf <= bit_buf >> D;
                    bits <= bits_after_out;
                    coeffs_done <= coeffs_done + 9'd1;

                    if ((coeffs_done + 9'd1) == 9'd256) begin
                        done <= 1'b1;
                        active <= 1'b0;
                        state <= WAIT;
                    end
                    else if (bits_after_out >= D[4:0]) begin
                        state <= OUT;
                    end
                    else begin
                        state <= WAIT;
                    end
                end

                default: state <= WAIT;
            endcase
        end
        else begin
            coeff_valid <= 1'b0;
        end
    end
endmodule
// Poly / Vector ALU: streams coefficient-wise addition or subtraction over n_polys * 256
module alu(
    input clk,
    input reset,
    input start,
    input op,
    input [2:0] n_polys,
    input [11:0] a_in,
    input [11:0] b_in,
    input in_valid,
    output ready,
    output reg [11:0] c_out,
    output reg out_valid,
    output reg done
    );

    localparam [1:0] IDLE = 2'd0;
    localparam [1:0] RUN = 2'd1;
    localparam [1:0] FIN = 2'd2;

    reg [1:0] state;
    reg op_r;
    reg [10:0] total;
    reg [10:0] count;
    reg [11:0] result;

    wire [11:0] add_r;
    wire [11:0] sub_r;

    assign ready = (state == RUN);

    mod_add u_add (
        .A(a_in),
        .B(b_in),
        .result(add_r)
    );

    mod_sub u_sub (
        .A(a_in),
        .B(b_in),
        .result(sub_r)
    );

    always @(*) begin
        if (op_r == 1'b0)
            result = add_r;
        else
            result = sub_r;
    end

    always @(posedge clk or posedge reset) begin
        if (reset) begin
            state <= IDLE;
            op_r <= 1'b0;
            total <= 11'd0;
            count <= 11'd0;
            c_out <= 12'd0;
            out_valid <= 1'b0;
            done <= 1'b0;
        end
        else begin
            out_valid <= 1'b0;
            done <= 1'b0;

            case (state)
                IDLE: begin
                    if (start) begin
                        op_r <= op;
                        count <= 11'd0;

                        if (n_polys == 3'd0)
                            total <= 11'd256;
                        else if (n_polys > 3'd4)
                            total <= 11'd1024;
                        else
                            total <= {n_polys, 8'd0};
                        state <= RUN;
                    end
                end

                RUN: begin
                    if (in_valid) begin
                        c_out <= result;
                        out_valid <= 1'b1;
                        count <= count + 11'd1;

                        if (count == (total - 11'd1))
                            state <= FIN;
                    end
                end

                FIN: begin
                    done <= 1'b1;
                    state <= IDLE;
                end

                default: state <= IDLE;
            endcase
        end
    end
endmodule


Overwriting kem.v


In [32]:
with open('kem.v', 'r') as f:
    lines = f.readlines()

# Remove lines 2417 to 2533 (inclusive) which contain the 'library' definition.
# Python lists are 0-indexed, so this corresponds to indices 2416 to 2532.
modified_lines = lines[:2416] + lines[2533:]

with open('kem.v', 'w') as f:
    f.writelines(modified_lines)

print("Removed 'library' definition from kem.v")

Removed 'library' definition from kem.v


In [34]:
import os

# Create the directory expected by the Verilog code
os.makedirs('src/memory', exist_ok=True)

# Writing the specific cryptographic twiddle factors provided by the user
twiddle_content = """8ED
A0B
B9A
714
5D5
58E
11F
0CA
C56
26E
629
0B6
3C2
84F
73F
5BC
23D
7D4
108
17F
9C4
5B2
6BF
C7F
A58
3F9
2DC
260
6FB
19B
C34
6DE
4C7
28C
AD9
3F7
7F4
5D3
BE7
6F9
204
CF9
BC1
A67
6AF
877
07E
5BD
9AC
CA7
BF2
33E
06B
774
C0A
94A
B73
3C1
71D
A2C
1C0
8D8
2A5
806
8B2
1AE
22B
34B
81E
367
60E
069
1A6
24B
0B1
C16
BDE
B35
626
675
C0B
30A
487
C6E
9F8
5CB
AA7
45F
6CB
284
999
15D
1A2
149
C65
CB6
331
449
25B
262
52A
7FC
748
180
842
C79
4C2
7CA
997
0DC
85E
686
860
707
803
31A
71B
9AB
99B
1DE
C95
BCD
3E4
3DF
3BE
74D
5F2
65C"""

with open('src/memory/twiddle.mem', 'w') as f:
    f.write(twiddle_content.strip())

print('Successfully updated src/memory/twiddle.mem with user provided values.')

Successfully updated src/memory/twiddle.mem with user provided values.


### Setting up the configuration

LibreLane requries you to configure any Flow before using it. This is done using
the `config` module.

For colaboratories, REPLs and other interactive environments where there is no
concrete Flow object, the Configuration may be initialized using `Config.interactive`,
which will automatically propagate the configuration to any future steps.

You can find the documentation for `Config.interactive` [here](https://librelane.readthedocs.io/en/latest/reference/api/config/index.html#librelane.config.Config.interactive).



In [36]:
from librelane.config import Config

Config.interactive(
    "kem",
    PDK="sky130A",
    CLOCK_PORT="clk",
    CLOCK_NET="clk",
    CLOCK_PERIOD=50,
    PRIMARY_GDSII_STREAMOUT_TOOL="klayout",
    MAX_TRANSITION_CONSTRAINT=1.5,
    MAX_FANOUT_CONSTRAINT=16
)


### Interactive Configuration
#### Initial Values

<br />

```yaml
CELL_BB_VERILOG_MODELS:
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/verilog/sky130_fd_sc_hd__blackbox.v
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/verilog/sky130_fd_sc_hd__blackbox_pp.v
CELL_CDLS:
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/cdl/sky130_fd_sc_hd.cdl
CELL_GDS:
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/gds/sky130_fd_sc_hd.gds
CELL_LEFS:
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/lef/sky130_fd_sc_hd.lef
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/lef/sky130_ef_sc_hd.lef
CELL_PAD_EXCLUDE:
- sky130_fd_sc_hd__tap*
- sky130_fd_sc_hd__decap*
- sky130_ef_sc_hd__decap*
- sky130_fd_sc_hd__fill*
CELL_SPICE_MODELS:
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/spice/sky130_ef_sc_hd__decap_12.spice
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/spice/sky130_ef_sc_hd__decap_20_12.spice
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/spice/sky130_ef_sc_hd__decap_40_12.spice
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/spice/sky130_ef_sc_hd__decap_60_12.spice
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/spice/sky130_ef_sc_hd__decap_80_12.spice
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/spice/sky130_ef_sc_hd__fill_12.spice
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/spice/sky130_ef_sc_hd__fill_2.spice
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/spice/sky130_ef_sc_hd__fill_4.spice
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/spice/sky130_ef_sc_hd__fill_8.spice
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/spice/sky130_fd_sc_hd.spice
CELL_VERILOG_MODELS:
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/verilog/primitives.v
- /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/verilog/sky130_fd_sc_hd.v
CLOCK_NET: clk
CLOCK_PERIOD: 50
CLOCK_PORT: clk
CLOCK_TRANSITION_CONSTRAINT: 0.15
CLOCK_UNCERTAINTY_CONSTRAINT: 0.25
DECAP_CELLS:
- sky130_fd_sc_hd__decap_3
DEFAULT_CORNER: nom_tt_025C_1v80
DEFAULT_MAX_TRAN: null
DESIGN_DIR: .
DESIGN_NAME: kem
DIE_AREA: null
DIODE_CELL: sky130_fd_sc_hd__diode_2/DIODE
ENDCAP_CELL: sky130_fd_sc_hd__decap_3
EXTRA_CDLS: null
EXTRA_EXCLUDED_CELLS: null
EXTRA_GDS: null
EXTRA_LEFS: null
EXTRA_LIBS: null
EXTRA_SPICE_MODELS: null
EXTRA_VERILOG_MODELS: null
FALLBACK_SDC: /content/librelane_ipynb/librelane/scripts/base.sdc
FILL_CELLS:
- sky130_fd_sc_hd__fill_2
- sky130_fd_sc_hd__fill_1
GND_NETS: null
GND_PIN: VGND
IO_DELAY_CONSTRAINT: 20
LIB:
  '*_ff_n40C_1v95':
  - /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/lib/sky130_fd_sc_hd__ff_n40C_1v95.lib
  '*_ss_100C_1v60':
  - /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/lib/sky130_fd_sc_hd__ss_100C_1v60.lib
  '*_tt_025C_1v80':
  - /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/lib/sky130_fd_sc_hd__tt_025C_1v80.lib
MACROS: null
MAX_CAPACITANCE_CONSTRAINT: 0.2
MAX_FANOUT_CONSTRAINT: 16
MAX_TRANSITION_CONSTRAINT: 1.5
OUTPUT_CAP_LOAD: 33.442
PAD_BONDPAD_HEIGHT: null
PAD_BONDPAD_NAME: null
PAD_BONDPAD_OFFSETS: null
PAD_BONDPAD_WIDTH: null
PAD_CDLS: null
PAD_CORNER: null
PAD_CORNER_SITE_NAME: null
PAD_EDGE_SPACING: 0
PAD_FAKE_SITES: null
PAD_FILLERS: null
PAD_GDS: null
PAD_LEFS: null
PAD_LIBS: null
PAD_PLACE_IO_TERMINALS: null
PAD_SITE_NAME: null
PAD_SPICE_MODELS: null
PAD_VERILOG_MODELS: null
PDK: sky130A
PDK_ROOT: /root/.ciel
PLACE_SITE: unithd
PNR_EXCLUDED_CELL_FILE: /root/.ciel/sky130A/libs.tech/openlane/sky130_fd_sc_hd/drc_exclude.cells
PRIMARY_GDSII_STREAMOUT_TOOL: klayout
RT_MAX_LAYER: met5
RT_MIN_LAYER: met1
SCL_GROUND_PINS:
- VGND
- VNB
SCL_POWER_PINS:
- VPWR
- VPB
STA_CORNERS:
- nom_tt_025C_1v80
- nom_ss_100C_1v60
- nom_ff_n40C_1v95
- min_tt_025C_1v80
- min_ss_100C_1v60
- min_ff_n40C_1v95
- max_tt_025C_1v80
- max_ss_100C_1v60
- max_ff_n40C_1v95
STD_CELL_LIBRARY: sky130_fd_sc_hd
SYNTH_BUFFER_CELL: sky130_fd_sc_hd__buf_2/A/X
SYNTH_CLK_DRIVING_CELL: null
SYNTH_DRIVING_CELL: sky130_fd_sc_hd__inv_2/Y
SYNTH_EXCLUDED_CELL_FILE: /root/.ciel/sky130A/libs.tech/openlane/sky130_fd_sc_hd/no_synth.cells
SYNTH_TIEHI_CELL: sky130_fd_sc_hd__conb_1/HI
SYNTH_TIELO_CELL: sky130_fd_sc_hd__conb_1/LO
TECH_LEFS:
  max_*: /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/techlef/sky130_fd_sc_hd__max.tlef
  min_*: /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/techlef/sky130_fd_sc_hd__min.tlef
  nom_*: /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/techlef/sky130_fd_sc_hd__nom.tlef
TIME_DERATING_CONSTRAINT: 5
TRISTATE_CELLS:
- sky130_fd_sc_hd__ebuf*
VDD_NETS: null
VDD_PIN: VPWR
WELLTAP_CELL: sky130_fd_sc_hd__tapvpwrvgnd_1
meta:
  flow: null
  librelane_version: 3.0.6
  step: null
  substituting_steps: null
  version: 1

```


### Running implementation steps

There are two ways to obtain LibreLane's built-in implementation steps:

* via directly importing from the `steps` module using its category:
    * `from librelane.steps import Yosys` then `Synthesis = Yosys.Synthesis`
* by using the step's id from the registry:
    * `from librelane.steps import Step` then `Synthesis = Step.factory.get("Yosys.Synthesis")`

You can find a full list of included steps here: https://librelane.readthedocs.io/en/latest/reference/step_config_vars.html

In [37]:
from librelane.steps import Step

* First, get the step (and display its help)...

In [38]:
Synthesis = Step.factory.get("Yosys.Synthesis")

Synthesis.display_help()



### Synthesis

```{eval-rst}

Performs synthesis and technology mapping on Verilog RTL files
using Yosys and ABC, emitting a netlist.

Some metrics will also be extracted and updated, namely:

* ``design__instance__count``
* ``design__instance_unmapped__count``
* ``design__instance__area``

Note that Yosys steps do not currently support gzipped standard cell dotlib
files. They are however supported for macros:

https://github.com/YosysHQ/yosys/issues/4830

```

#### Importing
```python
from librelane.steps.pyosys import Synthesis

# or

from librelane.steps import Step

Synthesis = Step.factory.get("Yosys.Synthesis")
```


#### Inputs and Outputs

| Inputs | Outputs |
| - | - |
|  | Verilog Netlist (.nl.v) |


#### Configuration Variables

| Variable Name | Type | Description | Default | 
| - | - | - | - | 
| `SYNTH_LATCH_MAP` <sup>PDK</sup> | Path? | A path to a file containing the latch mapping for Yosys. | `None` |
| `SYNTH_TRISTATE_MAP` <sup>PDK</sup> | Path? | A path to a file containing the tri-state buffer mapping for Yosys. | `None` |
| `SYNTH_CSA_MAP` <sup>PDK</sup> | Path? | A path to a file containing the carry-select adder mapping for Yosys. | `None` |
| `SYNTH_RCA_MAP` <sup>PDK</sup> | Path? | A path to a file containing the ripple-carry adder mapping for Yosys. | `None` |
| `SYNTH_FA_MAP` <sup>PDK</sup> | Path? | A path to a file containing the full adder mapping for Yosys. | `None` |
| `SYNTH_CLOCKGATE_MIN_WIDTH`  | int? | If set to a value, a group of flip-flops with size >= SYNTH_CLOCKGATE_MIN_WIDTH and an enable signal are clock-gated instead. | `None` |
| `SYNTH_CLOCKGATE_POSEDGE_ICG` <sup>PDK</sup> | str? | The integrated clock gate cell used for positive-edge flip-flops, in the format `<cell>/<active-high clock enable port>/<clk port>/<gated clk port>`. | `None` |
| `SYNTH_CLOCKGATE_NEGEDGE_ICG` <sup>PDK</sup> | str? | The integrated clock gate cell used for positive-edge flip-flops, in the format `<cell>/<active-high clock enable port>/<clk port>/<gated clk port>`. | `None` |
| `YOSYS_LOG_LEVEL`  | 'ALL'｜<br />'WARNING'｜<br />'ERROR' | Which log level for Yosys. At WARNING or higher, the initialization splash is also disabled. | `ALL` |
| `SYNTH_CORNER` <sup>PDK</sup> | str? | A fully qualified IPVT corner to use during synthesis. If unspecified, the value for `DEFAULT_CORNER` from the PDK will be used. | `None` |
| `SYNTH_SHOW`  | bool | Generate a graphviz DOT file for the design. This will fail on a completely empty design. | `False` |
| `SYNTH_CHECKS_ALLOW_TRISTATE`  | bool | Ignore multiple-driver warnings if they are connected to tri-state buffers on a best-effort basis. | `True` |
| `SYNTH_AUTONAME`  | bool | Generates names for netlist instances. This results in instance names that can be extremely long, but are more human-readable. | `False` |
| `SYNTH_STRATEGY`  | 'AREA 0'｜<br />'AREA 1'｜<br />'AREA 2'｜<br />'AREA 3'｜<br />'DELAY 0'｜<br />'DELAY 1'｜<br />'DELAY 2'｜<br />'DELAY 3'｜<br />'DELAY 4' | Strategies for abc logic synthesis and technology mapping. AREA strategies usually result in a more compact design, while DELAY strategies usually result in a design that runs at a higher frequency. Please note that there is no way to know which strategy is the best before trying them. | `AREA 0` |
| `SYNTH_ABC_BUFFERING`  | bool | Enables `abc` cell buffering. | `False` |
| `SYNTH_ABC_LEGACY_REFACTOR`  | bool | Replaces the ABC command `drf -l` with `refactor` which matches older versions of LibreLane but is more unstable. | `False` |
| `SYNTH_ABC_LEGACY_REWRITE`  | bool | Replaces the ABC command `drw -l` with `rewrite` which matches older versions of LibreLane but is more unstable. | `False` |
| `SYNTH_ABC_DFF`  | bool | Passes D-flipflop cells through ABC for optimization (which can for example, eliminate identical flip-flops). | `False` |
| `SYNTH_ABC_USE_MFS3`  | bool | Experimental: attempts a SAT-based remapping in all area and delay strategies before 'retime', which may improve PPA results. | `False` |
| `SYNTH_ABC_AREA_USE_NF`  | bool | Experimental: uses the &nf delay-based mapper with a very high value instead of the amap area mapper, which may be better in some scenarios at recovering area. | `False` |
| `SYNTH_DIRECT_WIRE_BUFFERING`  | bool | Enables inserting buffer cells for directly connected wires. | `True` |
| `SYNTH_SPLITNETS`  | bool | Splits multi-bit nets into single-bit nets. Easier to trace but may not be supported by all tools. | `True` |
| `SYNTH_SIZING`  | bool | Enables `abc` cell sizing (instead of buffering). | `False` |
| `SYNTH_HIERARCHY_MODE`  | 'flatten'｜<br />'deferred_flatten'｜<br />'keep' | Affects how hierarchy is maintained throughout and after synthesis. 'flatten' flattens it during and after synthesis. 'deferred_flatten' flattens it after synthesis. 'keep' never flattens it. Please note that when using the Slang plugin, you need to pass '--keep-hierarchy' to `SLANG_ARGUMENTS` separately. To keep the hierarchy partially, use one of the flattening options and set the 'keep_hierarchy' attribute on instances or modules via: `SYNTH_KEEP_HIERARCHY_INSTANCES`, `SYNTH_KEEP_HIERARCHY_MODULES` or `SYNTH_KEEP_HIERARCHY_MIN_COST`. | `flatten` |
| `SYNTH_KEEP_HIERARCHY_MIN_COST`  | int? | Sets the 'keep_hierarchy' attribute on modules where the gate count is estimated to exceed the specified threshold. This prevents larger modules from being flattened. This variable only affects the design when 'flatten' is called through `SYNTH_HIERARCHY_MODE`. | `None` |
| `SYNTH_KEEP_HIERARCHY_INSTANCES`  | List[str]? | A list of instances for which to set the 'keep_hierarchy' attribute. This variable only affects the design when 'flatten' is called through `SYNTH_HIERARCHY_MODE`. | `None` |
| `SYNTH_KEEP_HIERARCHY_MODULES`  | List[str]? | A list of modules for which to set the 'keep_hierarchy' attribute. This variable only affects the design when 'flatten' is called through `SYNTH_HIERARCHY_MODE`. | `None` |
| `SYNTH_SHARE_RESOURCES`  | bool | A flag that enables yosys to reduce the number of cells by determining shareable resources and merging them. | `True` |
| `SYNTH_ADDER_TYPE`  | 'YOSYS'｜<br />'FA'｜<br />'RCA'｜<br />'CSA' | Adder type to which the $add and $sub operators are mapped to.  Possible values are `YOSYS/FA/RCA/CSA`; where `YOSYS` refers to using Yosys internal adder definition, `FA` refers to full-adder structure, `RCA` refers to ripple carry adder structure, and `CSA` refers to carry select adder. | `YOSYS` |
| `SYNTH_EXTRA_MAPPING_FILE`  | Path? | Points to an extra techmap file for yosys that runs right after yosys `synth` before generic techmap. | `None` |
| `SYNTH_ELABORATE_ONLY`  | bool | "Elaborate" the design only without attempting any logic mapping. Useful when dealing with structural Verilog netlists. | `False` |
| `SYNTH_MUL_BOOTH`  | bool | Runs the booth pass as part of synthesis: See https://yosyshq.readthedocs.io/projects/yosys/en/latest/cmd/booth.html | `False` |
| `SYNTH_TIE_UNDEFINED`  | 'high'｜<br />'low' | Whether to tie undefined values low or high. Explicitly provide null if you wish to simply leave them undriven. | `low` |
| `SYNTH_WRITE_NOATTR`  | bool | If true, Verilog-2001 attributes are omitted from output netlists. Some utilities do not support attributes. | `True` |
| `SYNTH_NORMALIZE_SINGLE_BIT_VECTORS`  | bool | If true, vectors with the shape [0:0] are converted to normal wires in the netlist. If disabled, even one-width pins will be suffixed [0] in the layout when imported by most PnR tools. | `True` |
| `VERILOG_FILES`  | List[Path] | The paths of the design's Verilog files. | `None` |
| `VERILOG_DEFINES`  | List[str]? | Preprocessor defines for input Verilog files. | `None` |
| `VERILOG_POWER_DEFINE`  | str? | Specifies the name of the define used to guard power and ground connections in the input RTL. | `USE_POWER_PINS` |
| `VERILOG_INCLUDE_DIRS`  | List[Path]? | Specifies the Verilog `include` directories. | `None` |
| `SYNTH_PARAMETERS`  | List[str]? | Key-value pairs to be `chparam`ed in Yosys, in the format `key1=value1`. | `None` |
| `USE_SLANG`  | bool | Use the Slang frontend to process files, which has better SystemVerilog parsing capabilities but is not as battle-tested as the default Yosys friend. | `False` |
| `SLANG_ARGUMENTS`  | List[str]? | Pass arguments to the Slang frontend. | `None` |



* Then run it. Note you can pass step-specific configs using Python keyword
  arguments.

### Synthesis

We need to start by converting our high-level Verilog to one that just shows
the connections between small silicon patterns called "standard cells" in process
called Synthesis. We can do this by passing the Verilog files as a configuration
variable to `Yosys.Synthesis` as follows, then running it.

As this is the first step, we need to create an empty state and pass it to it.

In [2]:
from librelane.state import State
from librelane.steps import Step

Synthesis = Step.factory.get("Yosys.Synthesis")

synthesis = Synthesis(
    VERILOG_FILES=["./kem.v"],
    state_in=State(),
)
synthesis.start()

ModuleNotFoundError: No module named 'librelane'

In [16]:
log_file_path = "/content/librelane_run/2-yosys-synthesis/yosys-synthesis.log"
with open(log_file_path, 'r') as f:
    print(f.read())


 /----------------------------------------------------------------------------\
 |  yosys -- Yosys Open SYnthesis Suite                                       |
 |  Copyright (C) 2012 - 2026  Claire Xenia Wolf <claire@yosyshq.com>         |
 |  Distributed under an ISC-like license, type "license" to see terms        |
 \----------------------------------------------------------------------------/
 Yosys 0.62 (git sha1 7326bb7d6641500ecb285c291a54a662cb1e76cf, clang++ 21.1.2 -fPIC -O3)

1. Executing Liberty frontend: /root/.ciel/sky130A/libs.ref/sky130_fd_sc_hd/lib/sky130_fd_sc_hd__tt_025C_1v80.lib
Imported 428 cell types from liberty file.
[INFO] Using SDC file '/content/librelane_run/2-yosys-synthesis/synthesis.abc.sdc' for ABC…wtaf

2. Executing Verilog-2005 frontend: ./kem.v
Parsing SystemVerilog input from `./kem.v' to AST representation.
./kem.v:2417: ERROR: syntax error, unexpected TOK_ID



In [31]:
file_path = 'kem.v'
line_number = 2417

with open(file_path, 'r') as f:
    lines = f.readlines()

start_line = max(0, line_number - 3)
end_line = min(len(lines), line_number + 2)

for i in range(start_line, end_line):
    print(f'{i + 1}: {lines[i].strip()}')

2415: endmodule
2416: 
2417: 
2418: 
2419: module sky130_sram_1kbyte_1rw1r_32x256_8(


In [ ]:
display(synthesis)

### Floorplanning

Floorplanning does two things:

* Determines the dimensions of the final chip.
* Creates the "cell placement grid" which placed cells must be aligned to.
    * Each cell in the grid is called a "site." Cells can occupy multiple
      sites, with the overwhelming majority of cells occupying multiple sites
      by width, and some standard cell libraries supporting varying heights as well.

> Don't forget- you may call `display_help()` on any Step class to get a full
> list of configuration variables.


In [ ]:
Floorplan = Step.factory.get("OpenROAD.Floorplan")

floorplan = Floorplan(state_in=synthesis.state_out)
floorplan.start()

In [ ]:
display(floorplan)

### Tap/Endcap Cell Insertion

This places two kinds of cells on the floorplan:

* End cap/boundary cells: Added at the beginning and end of each row. True to
  their name, they "cap off" the core area of a design.
* Tap cells: Placed in a polka dot-ish fashion across the rows. Tap cells
  connect VDD to the nwell and the psubstrate to VSS, which the majority of cells
  do not do themselves to save area- but if you go long enough without one such
  connection you end up with the cell "latching-up"; i.e.; refusing to switch
  back to LO from HI.

  There is a maximum distance between tap cells enforced as part of every
  foundry process.

In [ ]:
TapEndcapInsertion = Step.factory.get("OpenROAD.TapEndcapInsertion")

tdi = TapEndcapInsertion(state_in=floorplan.state_out)
tdi.start()

In [ ]:
display(tdi)

### I/O Placement

This places metal pins at the edges of the design corresponding to the top level
inputs and outputs for your design. These pins act as the interface with other
designs when you integrate it with other designs.

In [ ]:
IOPlacement = Step.factory.get("OpenROAD.IOPlacement")

ioplace = IOPlacement(state_in=tdi.state_out)
ioplace.start()

In [ ]:
display(ioplace)

### Generating the Power Distribution Network (PDN)

This creates the power distribution network for your design, which is essentially
a plaid pattern of horizontal and vertical "straps" across the design that is
then connected to the rails' VDD and VSS (via the tap cells.)

You can find an explanation of how the power distribution network works at this
link: https://librelane.readthedocs.io/en/latest/usage/hardening_macros.html#pdn-generation

While we typically don't need to mess with the PDN too much, the SPM is a small
design, so we're going to need to make the plaid pattern formed by the PDN a bit
smaller.

In [ ]:
GeneratePDN = Step.factory.get("OpenROAD.GeneratePDN")

pdn = GeneratePDN(
    state_in=ioplace.state_out,
    FP_PDN_VWIDTH=2,
    FP_PDN_HWIDTH=2,
    FP_PDN_VPITCH=30,
    FP_PDN_HPITCH=30,
)
pdn.start()

In [ ]:
display(pdn)

### Global Placement

Global Placement is deciding on a fuzzy, non-final location for each of the cells,
with the aim of minimizing the distance between cells that are connected
together (more specifically, the total length of the not-yet-created wires that
will connect them).

As you will see in the `.display()` in the second cell below, the placement is
considered "illegal", i.e., not properly aligned with the cell placement grid.
This is addressed by "Detailed Placement", also referred to as "placement
legalization", which is the next step.

In [ ]:
GlobalPlacement = Step.factory.get("OpenROAD.GlobalPlacement")

gpl = GlobalPlacement(state_in=pdn.state_out)
gpl.start()

In [ ]:
display(gpl)

### Detailed Placement

This aligns the fuzzy placement from before with the grid, "legalizing" it.

In [ ]:
DetailedPlacement = Step.factory.get("OpenROAD.DetailedPlacement")

dpl = DetailedPlacement(state_in=gpl.state_out)
dpl.start()

In [ ]:
display(dpl)

### Clock Tree Synthesis (CTS)

With the cells now having a final placement, we can go ahead and create what
is known as the clock tree, i.e., the hierarchical set of buffers used
for clock signal to minimize what is known as "clock skew"- variable delay
of the clock cycle from register to register because of factors such as metal
wire length, clock load (number of gates connected to the same clock buffer,)
et cetera.

The CTS step creates the cells and places the between the gaps in the detailed
placement above.

In [ ]:
CTS = Step.factory.get("OpenROAD.CTS")

cts = CTS(state_in=dpl.state_out)
cts.start()

In [ ]:
display(cts)

### Global Routing

Global routing "plans" the routes the wires between two gates (or gates and
I/O pins/the PDN) will take. The results of global routing (which are called
"routing guides") are stored in internal data structures and have no effect on
the actual design, so there is no `display()` statement.

In [ ]:
GlobalRouting = Step.factory.get("OpenROAD.GlobalRouting")

grt = GlobalRouting(state_in=cts.state_out)
grt.start()

### Detailed Routing

Detailed routing uses the guides from Global Routing to actually create wires
on the metal layers and connect the gates, making the connections finally physical.

This is typically the longest step in the flow.

In [ ]:
DetailedRouting = Step.factory.get("OpenROAD.DetailedRouting")

drt = DetailedRouting(state_in=grt.state_out)
drt.start()

In [ ]:
display(drt)

### Fill Insertion

Finally, as we're done placing all the essential cells, the only thing left to
do is fill in the gaps.

We prioritize the use of decap (decoupling capacitor) cells, which
further supports the power distribution network, but when there aren't any
small enough cells, we just use regular fill cells.

In [ ]:
FillInsertion = Step.factory.get("OpenROAD.FillInsertion")

fill = FillInsertion(state_in=drt.state_out)
fill.start()

In [ ]:
display(fill)

### Parasitics Extraction a.k.a. Resistance/Capacitance Extraction (RCX)

This step does not alter the design- rather, it computes the
[Parasitic elements](https://en.wikipedia.org/wiki/Parasitic_element_(electrical_networks))
of the circuit, which have an effect of timing, as we prepare to do the final
timing analysis.

The parasitic elements are saved in the **Standard Parasitics Exchange Format**,
or SPEF. LibreLane creates a SPEF file for each interconnect corner as described in
the [Corners and STA](https://librelane.readthedocs.io/en/latest/usage/corners_and_sta.html)
section of the documentation.

In [ ]:
RCX = Step.factory.get("OpenROAD.RCX")

rcx = RCX(state_in=fill.state_out)
rcx.start()

### Static Timing Analysis (Post-PnR)

STA is a process that verifies that a chip meets certain constraints on clock
and data timings to run at its rated clock speed. See [Corners and STA](https://librelane.readthedocs.io/en/latest/usage/corners_and_sta.html)
in the documentation for more info.

---

This step generates two kinds of files:
* `.lib`: Liberty™-compatible Library files. Can be used to do static timing
  analysis when creating a design with this design as a sub-macro.
* `.sdf`: Standard Delay Format. Can be used with certain simulation software
  to do *dynamic* timing analysis.

Unfortunately, the `.lib` files coming out of LibreLane right now are not super
reliable for timing purposes and are only provided for completeness.

When using LibreLane-created macros withing other designs, it is best to use the
macro's final netlist and extracted parasitics instead.

In [ ]:
STAPostPNR = Step.factory.get("OpenROAD.STAPostPNR")

sta_post_pnr = STAPostPNR(state_in=rcx.state_out)
sta_post_pnr.start()

### Stream-out

Stream-out is the process of converting the designs from the abstract formats
using during floorplanning, placement and routing into a concrete format called
GDSII (lit. Graphic Design System 2), which is the final file that is then sent
for fabrication.

In [ ]:
StreamOut = Step.factory.get("KLayout.StreamOut")

gds = StreamOut(state_in=sta_post_pnr.state_out)
gds.start()

In [ ]:
display(gds)

### Design Rule Checks (DRC)

DRC determines that the final layout does not violate any of the rules set by
the foundry to ensure the design is actually manufacturable- for example,
not enough space between two wires, *too much* space between tap cells, and so
on.

A design not passing DRC will typically be rejected by the foundry, who
also run DRC on their side.

In [ ]:
DRC = Step.factory.get("Magic.DRC")

drc = DRC(state_in=gds.state_out)
drc.start()

### SPICE Extraction for Layout vs. Schematic Check

This step tries to reconstruct a SPICE netlist from the GDSII file, so it can
later be used for the **Layout vs. Schematic** (LVS) check.

In [ ]:
SpiceExtraction = Step.factory.get("Magic.SpiceExtraction")

spx = SpiceExtraction(state_in=drc.state_out)
spx.start()

### Layout vs. Schematic (LVS)

A comparison between the final Verilog netlist (from PnR) and the final
SPICE netlist (extracted.)

This check effectively compares the physically implemented circuit to the final
Verilog netlist output by OpenROAD.

The idea is, if there are any disconnects, shorts or other mismatches in the
physical implementation that do not exist in the logical view of the design,
they would be caught at this step.

Common issues that result in LVS violations include:
* Lack of fill cells or tap cells in the design
* Two unrelated signals to be shorted, or a wire to be disconnected (most
  commonly seen with misconfigured PDN)

Chips with LVS errors are typically dead on arrival.

In [ ]:
LVS = Step.factory.get("Netgen.LVS")

lvs = LVS(state_in=spx.state_out)
lvs.start()